In [1]:
# # Merge & Validate Well-FOV Feature Parquets — All Patients
#
# ## Purpose
# Same QC as `merge_well_fov_features_single_patient.ipynb`, but across **every patient**
# listed in `data/patient_IDs.txt` instead of one hardcoded patient. For every well-FOV of
# every patient, this attempts to merge the per-compartment x channel x feature-type
# parquet files into a single feature space per compartment, and flags every place that
# could silently go wrong: missing/unreadable files, missing merge keys, duplicate
# `object_id`s, merge blow-ups, an Organoid merge that fails outright due to an empty
# input file, object-ID misalignment across single-cell compartments, source segmentation
# masks that no longer match the extracted features, and profiles containing any
# `object_id` greater than 255.
#
# ## Why this is structured differently from the single-patient version
# Across all 13 patients this dataset has roughly **680,000** feature parquet files, some
# with 700+ feature columns (e.g. SAMMed3D). Reading every column of every file the way
# the single-patient notebook does would take hours. This notebook only needs `object_id`
# and `image_set` to do every check below (including the Organoid and source-mask checks
# it shares with the single-patient notebook), so it:
# 1. Checks each file's **schema** first (no data read) to catch missing merge keys cheaply.
# 2. Reads only the `object_id`/`image_set` columns (parquet column projection) instead of
#    the full file.
# 3. Reads files **in parallel** with a thread pool, since this is I/O-bound (many small
#    files) rather than CPU-bound.
#
# This still takes on the order of **20-30 minutes** for the full dataset. To test on a
# subset first, edit `PATIENTS` in the constants cell below to a shorter list.
#
# ## Inputs
# - `data/patient_IDs.txt` — the list of patients to process
# - `data/{patient}/extracted_features/{well_fov}/*.parquet` for each patient
#   - Expected filename format: `{Compartment}_{Channel}_{FeatureType}_{Processor}_features.parquet`
#   - Each file is expected to contain `object_id` and `image_set` columns
# - `data/{patient}/segmentation_masks/{well_fov}/*_mask.tiff` for each patient
#   - Read directly (not through the extracted features), and only for well-FOV/compartment
#     combinations implicated in a merge-related issue, to check whether an object-ID
#     discrepancy traces back to the segmentation mask itself
#
# ## Outputs (written to `3.cellprofiling/logs/`, one row per patient+well-FOV/issue)
# - `well_fov_feature_merge_summary_all_patients.csv`
# - `well_fov_feature_merge_issues_all_patients.csv`
# - `well_fov_feature_merge_report_all_patients.md`
#
# ## What counts as an issue here
# - A file that fails to read, or is missing the `object_id`/`image_set` merge keys
# - A well-FOV with **zero** parquet files (not yet processed at all) — tracked separately
#   from a partial file-count mismatch, since there's nothing to merge
# - A well-FOV whose file count differs from *that patient's own* expected count (each
#   patient can have a different channel/feature-type combination, so the expected count
#   is computed per patient as the mode of its well-FOVs' file counts, not hardcoded)
# - Duplicate `object_id` values within a single feature file
# - A within-compartment merge whose row count exceeds the union of `object_id` values
#   across that compartment's input files (a "blow-up" from a duplicated merge key)
# - An Organoid merge that fails outright because one of its input files is an empty
#   (0-row) profile — a known dtype-clash symptom, reclassified from a generic
#   `merge_error` — this is still a real bug and counts as an issue
# - **Within a compartment, any `object_id` across all of that compartment's input files
#   is greater than 255** — small sequential IDs are the expected "sequential" scheme, so
#   any value above 255 suggests that well-FOV/compartment is using the "z_slice_global"
#   ID scheme (IDs that encode a z-slice offset) instead
# - Object IDs that disagree across the single-cell compartments (`Nuclei`, `Cell`,
#   `Cytoplasm`, `Nucleocentric`) for a well-FOV as a whole. `Organoid` is excluded from
#   this comparison — it's a distinct feature space (one row per whole organoid, not per
#   cell) and is never expected to align with the single-cell compartments
# - A segmentation mask TIFF, read directly, whose unique object IDs don't match the
#   object IDs present in the extracted feature files for that compartment — checked for
#   every well-FOV/compartment implicated in one of the merge-related issues above, to
#   distinguish a **stale/corrupted source mask** (re-segmented/overwritten after
#   featurization last ran) from a bug in the merge/extraction code itself
#
# **Not** counted as an issue: a well-FOV with no readable Organoid feature files, or an
# Organoid merge that produces zero rows. Some well-FOVs legitimately have no organoids,
# so an empty Organoid profile is tracked and reported on its own (see
# `organoid_profiles_empty` in the summary and the Organoid section of the report) but
# does **not** count toward `n_issues` and does **not** trigger the source-mask
# consistency check on its own.
#

In [2]:
import json
import os
import pathlib
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from functools import reduce

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import tifffile
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
# Segmentation masks are only ever present on the NAS mount, not synced locally, so
# the NAS-resolved path is kept separately before profile_base_dir is overridden below.
segmentation_masks_base_dir = profile_base_dir
profile_base_dir = root_dir  # default to root_dir instead of NAS

In [3]:
output_features_subparent_name = "extracted_features"
mask_subparent_name = "segmentation_masks"
logs_dir = pathlib.Path(f"{root_dir}/3.cellprofiling/logs").resolve(strict=True)

MERGE_KEYS = ["object_id", "image_set"]
COMPARTMENTS = ["Organoid", "Nuclei", "Cell", "Cytoplasm", "Nucleocentric"]
SINGLE_CELL_COMPARTMENTS = ["Nuclei", "Cell", "Cytoplasm", "Nucleocentric"]
# Nucleocentric features are extracted against the Nuclei mask (see
# scripts/nucleo_centric_featurization.py), so it shares Nuclei\'s source mask file.
COMPARTMENT_MASK_FILENAME = {
    "Organoid": "organoid_mask.tiff",
    "Nuclei": "nuclei_mask.tiff",
    "Cell": "cell_mask.tiff",
    "Cytoplasm": "cytoplasm_mask.tiff",
    "Nucleocentric": "nuclei_mask.tiff",
}
# A profile\'s object_id > 255 is the signature of the "z_slice_global" ID scheme
# (ids encode a z-slice offset), vs. small sequential ids (1, 2, 3, ...) from the
# "sequential" scheme.
OBJECT_ID_SCHEME_THRESHOLD = 255
# Issue types that describe a legitimately-empty Organoid profile rather than a real
# merge/extraction bug — logged for visibility, but excluded from the n_issues/error
# counts and from triggering the source-mask consistency check.
NON_ERROR_ISSUE_TYPES = {"empty_organoid_profiles", "organoid_object_ids_empty"}
N_READ_WORKERS = 16

PATIENTS = pd.read_csv(
    pathlib.Path(f"{profile_base_dir}/data/patient_IDs.txt").resolve(strict=True),
    header=None,
    names=["patient_id"],
).patient_id.tolist()
# For a quick test run, uncomment and shorten:
# PATIENTS = PATIENTS[:1]
PATIENTS


# ## Discover well-FOV directories for every patient
# Every subdirectory of a patient's `extracted_features` is a well-FOV except
# `run_stats`, which holds per-run diagnostic parquets rather than merged feature files.
# The corresponding `segmentation_masks` directory (on the NAS mount) is recorded per
# patient too, for the source-mask consistency check later on.

['NF0014_T1',
 'NF0014_T2',
 'NF0016_T1',
 'NF0018_T6',
 'NF0021_T1',
 'NF0030_T1',
 'NF0035_T1',
 'NF0037_T1',
 'NF0037_T1_CQ1',
 'NF0040_T1',
 'NF0055_T1',
 'SARCO219_T2',
 'SARCO361_T1']

In [4]:
well_fov_dirs_by_patient = {}
segmentation_masks_dir_by_patient = {}
for patient in PATIENTS:
    patient_dir = pathlib.Path(
        f"{profile_base_dir}/data/{patient}/{output_features_subparent_name}"
    )
    if not patient_dir.is_dir():
        print(f"Skipping {patient}: no {output_features_subparent_name} directory")
        continue
    well_fov_dirs_by_patient[patient] = sorted(
        d for d in patient_dir.iterdir() if d.is_dir() and d.name != "run_stats"
    )
    segmentation_masks_dir_by_patient[patient] = pathlib.Path(
        f"{segmentation_masks_base_dir}/data/{patient}/{mask_subparent_name}"
    )

n_well_fovs_total = sum(len(v) for v in well_fov_dirs_by_patient.values())
print(f"{len(well_fov_dirs_by_patient)} patients, {n_well_fovs_total} well-FOVs total")


# ## Build the full file list, then read `object_id`/`image_set` in parallel
#
# Each task reads a file's schema first (cheap) to catch missing merge keys without a
# data read, then projects to just the two key columns. Errors (corrupt/zero-byte files,
# missing keys) are captured per file rather than raising.

13 patients, 4187 well-FOVs total


In [5]:
def parse_feature_filename(path: pathlib.Path) -> dict:
    """Expected format: {Compartment}_{Channel}_{FeatureType}_{Processor}_features.parquet"""
    parts = path.stem.split("_")
    return {
        "compartment": parts[0],
        "channel": parts[1] if len(parts) > 1 else None,
        "feature_type": parts[2] if len(parts) > 2 else None,
        "processor": parts[-2] if len(parts) > 1 else None,
    }

In [6]:
def read_key_columns(task: tuple) -> dict:
    """Read just object_id/image_set (+ mtime) for one file, capturing any error."""
    patient, well_fov, path = task
    result = {
        "patient": patient,
        "well_fov": well_fov,
        "file_name": path.name,
        "df": None,
        "mtime": None,
        "error_kind": None,
        "error_detail": None,
    }
    try:
        schema_cols = pq.ParquetFile(path).schema.names
    except Exception as e:
        result["error_kind"] = "read_error"
        result["error_detail"] = str(e)
        return result
    missing_keys = [k for k in MERGE_KEYS if k not in schema_cols]
    if missing_keys:
        result["error_kind"] = "missing_merge_keys"
        result["error_detail"] = f"missing {missing_keys}"
        return result
    try:
        result["df"] = pd.read_parquet(path, columns=MERGE_KEYS)
        result["mtime"] = path.stat().st_mtime
    except Exception as e:
        result["error_kind"] = "read_error"
        result["error_detail"] = str(e)
        result["df"] = None
    return result


# ## Read a segmentation mask's object IDs directly
# Used only for the source-mask consistency check below, and only for the specific
# well-FOV/compartment combinations implicated in a merge-related issue — reading every
# mask TIFF for every well-FOV upfront would be far more I/O than this notebook needs.

In [7]:
def get_mask_object_ids(mask_path: pathlib.Path) -> set | None:
    """Unique non-background object IDs read directly from a segmentation mask TIFF.

    Returns None if the mask file does not exist, so callers can distinguish a
    missing mask from a mask with zero detected objects.
    """
    if not mask_path.exists():
        return None
    mask = tifffile.imread(mask_path)
    return set(np.unique(mask).tolist()) - {0}

In [8]:
all_tasks = [
    (patient, well_fov_dir.name, f)
    for patient, well_fov_dirs in well_fov_dirs_by_patient.items()
    for well_fov_dir in well_fov_dirs
    for f in well_fov_dir.glob("*.parquet")
]
print(f"{len(all_tasks)} files to read across {len(well_fov_dirs_by_patient)} patients")

with ThreadPoolExecutor(max_workers=N_READ_WORKERS) as pool:
    file_results = list(
        tqdm.tqdm(
            pool.map(read_key_columns, all_tasks),
            total=len(all_tasks),
            desc="Reading object_id/image_set columns",
        )
    )


# ## Group per-file results by (patient, well-FOV, compartment)
# Also compute, per patient, the *expected* file count as the mode of that patient's
# well-FOVs' file counts (excluding well-FOVs with zero files) — each patient can have a
# different channel/feature-type combination, so a single hardcoded expected count
# (as used in the single-patient notebook) doesn't generalize.

417495 files to read across 13 patients


Reading object_id/image_set columns:   0%|          | 0/417495 [00:00<?, ?it/s]

In [9]:
files_by_well_fov = defaultdict(list)
for r in file_results:
    files_by_well_fov[(r["patient"], r["well_fov"])].append(r)

file_counts_by_patient = defaultdict(list)
for (patient, well_fov), results in files_by_well_fov.items():
    file_counts_by_patient[patient].append(len(results))

expected_n_files_by_patient = {
    patient: (
        pd.Series([c for c in counts if c > 0]).mode().iloc[0]
        if any(c > 0 for c in counts)
        else 0
    )
    for patient, counts in file_counts_by_patient.items()
}
pd.Series(expected_n_files_by_patient, name="expected_n_files").sort_index()


# ## Per-well-FOV merge attempt
#
# For each (patient, well-FOV):
# 1. Route each file's cached read result by compartment (files with a read/schema error
#    are logged and excluded from merging).
# 2. Within each compartment, outer-merge all its files on `object_id` + `image_set`
#    (outer, not left, so a merge error can never silently drop rows without being counted).
# 3. Flag a merge "blow-up" if the merged row count exceeds the union of `object_id`
#    values across that compartment's input files. For `Organoid` specifically, also
#    flag when the merge itself fails because an input file is an empty (0-row) profile
#    (a known dtype-clash symptom, reclassified from a generic `merge_error`). A well-FOV
#    with no Organoid files at all, or an Organoid merge that produces zero rows, is
#    tracked separately and does **not** count as an issue — some well-FOVs legitimately
#    have no organoids.
# 4. Within each compartment, flag when **any** `object_id` across all of that
#    compartment's input files is greater than 255 — small sequential IDs are the
#    expected scheme, so any value above 255 suggests the "z_slice_global" ID scheme is
#    in play for that well-FOV/compartment instead.
# 5. Compare object-ID sets across the single-cell compartments to catch alignment drift.
# 6. For every compartment implicated in a merge-related issue above (a blow-up, a
#    duplicate `object_id`, a failed Organoid merge, an object ID over 255, or a
#    cross-compartment misalignment), read that compartment's segmentation mask TIFF
#    directly and compare its actual unique object IDs against the object IDs present in
#    the extracted feature files. A mismatch traces the discrepancy back to the
#    segmentation mask on disk rather than to a bug in the merge/extraction code.

NF0014_T1        101
NF0014_T2        101
NF0016_T1        101
NF0018_T6        101
NF0021_T1        101
NF0030_T1        101
NF0035_T1        101
NF0037_T1        101
NF0037_T1_CQ1    101
NF0040_T1        101
NF0055_T1        101
SARCO219_T2      101
SARCO361_T1      101
Name: expected_n_files, dtype: int64

In [10]:
def process_well_fov(
    patient: str,
    well_fov: str,
    expected_n_files: int,
    segmentation_masks_dir: pathlib.Path,
) -> tuple[dict, list[dict]]:
    """Attempt to merge one well-FOV's cached feature-file reads per compartment."""
    results = files_by_well_fov[(patient, well_fov)]
    issues = []
    implicated_compartments = set()

    def log_issue(kind, detail, compartment=None):
        issues.append(
            {
                "patient": patient,
                "well_fov": well_fov,
                "issue_type": kind,
                "detail": detail,
            }
        )
        if compartment is not None:
            implicated_compartments.add(compartment)

    if len(results) == 0:
        log_issue("no_files_found", "0 parquet files found for this well-FOV")
        return (
            {
                "patient": patient,
                "well_fov": well_fov,
                "n_files_found": 0,
                "n_files_expected": expected_n_files,
                "file_count_mismatch": False,
                "no_files_found": True,
                "n_issues": 0,
                "n_read_errors": 0,
                "n_merge_errors": 0,
                "n_merge_blowups": 0,
                "n_duplicate_object_id_files": 0,
                "n_object_id_exceeds_threshold": 0,
                "object_ids_aligned_across_compartments": None,
                "misalignment_root_cause": None,
                "organoid_profiles_empty": None,
                "organoid_object_ids_empty": None,
                "n_organoid_empty_profile_merge_errors": 0,
                "n_source_mask_mismatches": 0,
                "n_source_mask_missing": 0,
                "compartments_present": [],
                "compartment_row_counts": {},
            },
            issues,
        )

    if len(results) != expected_n_files:
        log_issue(
            "file_count_mismatch",
            f"found {len(results)} files, expected {expected_n_files} "
            f"(this patient's mode)",
        )

    per_compartment_dfs = {c: [] for c in COMPARTMENTS}
    for r in results:
        if r["error_kind"] is not None:
            log_issue(r["error_kind"], f"{r['file_name']}: {r['error_detail']}")
            continue
        meta = parse_feature_filename(pathlib.Path(r["file_name"]))
        compartment = meta["compartment"]
        if compartment not in COMPARTMENTS:
            log_issue(
                "unknown_compartment", f"{r['file_name']}: compartment '{compartment}'"
            )
            continue
        df = r["df"]
        if df["object_id"].duplicated().any():
            n_dupes = int(df["object_id"].duplicated().sum())
            log_issue(
                "duplicate_object_id_in_file",
                f"{r['file_name']}: {n_dupes} duplicate object_id rows",
                compartment=compartment,
            )
        per_compartment_dfs[compartment].append((r["file_name"], df, r["mtime"]))

    # Organoid profiles are expected for every well-FOV (one row per organoid, unlike
    # the per-cell compartments), but some well-FOVs legitimately have none. Track it
    # for visibility without a `compartment=` kwarg — this is deliberately NOT counted
    # as an error and does NOT trigger the source-mask consistency check on its own.
    if not per_compartment_dfs["Organoid"]:
        log_issue(
            "empty_organoid_profiles",
            f"no readable Organoid feature files for {well_fov}; Organoid merge skipped",
        )

    compartment_shapes = {}
    for compartment, items in per_compartment_dfs.items():
        if not items:
            continue
        names = [n for n, _, _ in items]
        dfs = [d for _, d, _ in items]
        union_ids = set().union(*(set(d["object_id"]) for d in dfs))
        union_n_objects = len(union_ids)

        if union_ids and max(union_ids) > OBJECT_ID_SCHEME_THRESHOLD:
            log_issue(
                "object_id_exceeds_threshold",
                f"{compartment}: object_id values across {len(dfs)} input file(s) "
                f"include a maximum of {max(union_ids)}, greater than "
                f"{OBJECT_ID_SCHEME_THRESHOLD} — likely the 'z_slice_global' ID "
                f"scheme rather than small sequential IDs for this profile",
                compartment=compartment,
            )

        try:
            merged = reduce(
                lambda left, right: pd.merge(left, right, on=MERGE_KEYS, how="outer"),
                dfs,
            )
        except Exception as e:
            # A merge failure on the Organoid compartment that complains about a
            # dtype clash on the merge key is a known symptom of one of the input
            # files being an empty (0-row) profile: pandas infers float64 for an
            # empty object_id column, which then conflicts with the object/int
            # dtype of the populated Organoid files during the outer join. Detect
            # that root cause directly (rather than string-matching the pandas
            # error) and reclassify it as an Organoid-specific empty-profile issue
            # instead of a generic merge_error.
            empty_names = (
                [n for n, d, _ in items if len(d) == 0]
                if compartment == "Organoid"
                else []
            )
            if compartment == "Organoid" and empty_names:
                log_issue(
                    "organoid_empty_profile_merge_error",
                    f"Organoid merge failed because {empty_names} contain 0 rows "
                    f"(an empty object_id column typed as float64 conflicts with "
                    f"the object/int dtype in the populated Organoid files during "
                    f"merge). Original error: {e}",
                    compartment="Organoid",
                )
                continue
            log_issue("merge_error", f"{compartment} ({names}): {e}")
            continue
        compartment_shapes[compartment] = merged.shape
        if merged.shape[0] > union_n_objects:
            log_issue(
                "merge_blowup",
                f"{compartment}: merged to {merged.shape[0]} rows, but the union of "
                f"object_id values across its {len(dfs)} input files is only "
                f"{union_n_objects} — a merge key was duplicated somewhere",
                compartment=compartment,
            )
        # Organoid files can exist and read successfully but still carry zero
        # object_id rows (e.g. an upstream step wrote an empty table). Distinct
        # from the "no Organoid files at all" case checked above — this one only
        # shows up after merging, since union_n_objects is also 0 in this case so
        # it would not otherwise trip the merge-blowup check. Also not counted as
        # an error, for the same reason as the "no Organoid files" case.
        if compartment == "Organoid" and merged.shape[0] == 0:
            log_issue(
                "organoid_object_ids_empty",
                f"Organoid files present ({names}) but merged frame has 0 rows — "
                f"no object_id values found in any Organoid feature file for "
                f"{well_fov}",
            )

    id_sets = {
        c: set(pd.concat([d for _, d, _ in per_compartment_dfs[c]])["object_id"])
        for c in SINGLE_CELL_COMPARTMENTS
        if per_compartment_dfs[c]
    }
    aligned = None
    misaligned_compartments = set()
    if len(id_sets) > 1:
        reference_compartment, reference_ids = next(iter(id_sets.items()))
        aligned = True
        for compartment, ids in id_sets.items():
            if ids != reference_ids:
                aligned = False
                log_issue(
                    "object_id_misalignment",
                    f"{compartment} vs {reference_compartment}: "
                    f"only-in-{compartment}={sorted(ids - reference_ids)[:10]}, "
                    f"only-in-{reference_compartment}={sorted(reference_ids - ids)[:10]}",
                    compartment=compartment,
                )
                implicated_compartments.add(reference_compartment)
                misaligned_compartments.add(compartment)
                misaligned_compartments.add(reference_compartment)

    # For every compartment implicated in a merge-related issue above, read its
    # segmentation mask TIFF directly once (cached in mask_ids_by_compartment so the
    # image-level root-cause check below doesn't re-read the same file), then check
    # whether the mask's object IDs match the extracted feature object IDs. A
    # mismatch here means the mask itself is stale/corrupted relative to the
    # features (or vice versa) — not a bug in this merge/QC logic.
    mask_ids_by_compartment = {}
    for compartment in sorted(implicated_compartments):
        mask_path = (
            segmentation_masks_dir / well_fov / COMPARTMENT_MASK_FILENAME[compartment]
        )
        mask_ids = get_mask_object_ids(mask_path)
        if mask_ids is None:
            log_issue(
                "source_mask_missing",
                f"{compartment}: expected mask at {mask_path}",
            )
            continue
        mask_ids_by_compartment[compartment] = mask_ids
        feature_ids = (
            set().union(
                *(set(d["object_id"]) for _, d, _ in per_compartment_dfs[compartment])
            )
            if per_compartment_dfs[compartment]
            else set()
        )
        only_in_mask = mask_ids - feature_ids
        only_in_features = feature_ids - mask_ids
        if only_in_mask or only_in_features:
            log_issue(
                "source_mask_object_id_mismatch",
                f"{compartment}: mask has {len(mask_ids)} object IDs, extracted "
                f"features have {len(feature_ids)} — only-in-mask="
                f"{sorted(only_in_mask)[:10]}, only-in-features="
                f"{sorted(only_in_features)[:10]} — the segmentation mask on disk "
                f"does not match the extracted features, so the source mask is the "
                f"likely cause (re-segmented/overwritten after features were last "
                f"extracted) rather than a merge/extraction bug",
            )

    # Root-cause the cross-compartment object_id_misalignment above: is it already
    # present in the segmentation masks themselves (image-level — e.g. the Nuclei
    # and Cell masks were built from different object-ID assignments for the same
    # image), or do the masks actually agree with each other and the discrepancy
    # only shows up in the extracted feature tables (feature-extraction-level — a
    # bug in how object_id was written out per compartment during featurization)?
    # Every misaligned compartment's mask was already read into
    # mask_ids_by_compartment by the source-mask check above, since misaligned
    # compartments are always added to implicated_compartments.
    misalignment_root_cause = None
    if misaligned_compartments:
        available_mask_ids = {
            c: mask_ids_by_compartment[c]
            for c in misaligned_compartments
            if c in mask_ids_by_compartment
        }
        if len(available_mask_ids) < len(misaligned_compartments):
            misalignment_root_cause = "unknown_mask_missing"
        else:
            ref_compartment, ref_mask_ids = next(iter(available_mask_ids.items()))
            masks_agree = all(
                ids == ref_mask_ids for ids in available_mask_ids.values()
            )
            if not masks_agree:
                misalignment_root_cause = "image_level"
                for compartment, ids in available_mask_ids.items():
                    if ids != ref_mask_ids:
                        log_issue(
                            "object_id_misalignment_traced_to_image",
                            f"{compartment} vs {ref_compartment}: the segmentation "
                            f"masks themselves disagree on object IDs — "
                            f"only-in-{compartment}-mask="
                            f"{sorted(ids - ref_mask_ids)[:10]}, "
                            f"only-in-{ref_compartment}-mask="
                            f"{sorted(ref_mask_ids - ids)[:10]} — this well-FOV's "
                            f"object_id misalignment originates in the source "
                            f"masks/images, not in feature extraction or merging",
                            compartment=compartment,
                        )
            else:
                misalignment_root_cause = "feature_extraction_level"
                log_issue(
                    "object_id_misalignment_traced_to_features",
                    f"segmentation masks for {sorted(available_mask_ids)} agree "
                    f"with each other on object IDs, but the extracted feature "
                    f"tables for these compartments disagree — the misalignment "
                    f"was introduced during feature extraction or merging, not in "
                    f"the source masks/images",
                )

    summary = {
        "patient": patient,
        "well_fov": well_fov,
        "n_files_found": len(results),
        "n_files_expected": expected_n_files,
        "file_count_mismatch": len(results) != expected_n_files,
        "no_files_found": False,
        "n_issues": sum(
            1 for i in issues if i["issue_type"] not in NON_ERROR_ISSUE_TYPES
        ),
        "n_read_errors": sum(i["issue_type"] == "read_error" for i in issues),
        "n_merge_errors": sum(i["issue_type"] == "merge_error" for i in issues),
        "n_merge_blowups": sum(i["issue_type"] == "merge_blowup" for i in issues),
        "n_duplicate_object_id_files": sum(
            i["issue_type"] == "duplicate_object_id_in_file" for i in issues
        ),
        "n_object_id_exceeds_threshold": sum(
            i["issue_type"] == "object_id_exceeds_threshold" for i in issues
        ),
        "object_ids_aligned_across_compartments": aligned,
        "misalignment_root_cause": misalignment_root_cause,
        "organoid_profiles_empty": any(
            i["issue_type"]
            in (
                "empty_organoid_profiles",
                "organoid_object_ids_empty",
                "organoid_empty_profile_merge_error",
            )
            for i in issues
        ),
        "organoid_object_ids_empty": any(
            i["issue_type"] == "organoid_object_ids_empty" for i in issues
        ),
        "n_organoid_empty_profile_merge_errors": sum(
            i["issue_type"] == "organoid_empty_profile_merge_error" for i in issues
        ),
        "n_source_mask_mismatches": sum(
            i["issue_type"] == "source_mask_object_id_mismatch" for i in issues
        ),
        "n_source_mask_missing": sum(
            i["issue_type"] == "source_mask_missing" for i in issues
        ),
        "compartments_present": sorted(compartment_shapes.keys()),
        "compartment_row_counts": {c: s[0] for c, s in compartment_shapes.items()},
    }
    return summary, issues

In [11]:
summaries = []
all_issues = []
well_fov_keys = [
    (patient, well_fov_dir.name)
    for patient, well_fov_dirs in well_fov_dirs_by_patient.items()
    for well_fov_dir in well_fov_dirs
]
for patient, well_fov in tqdm.tqdm(
    well_fov_keys, desc="Merging all patients' well-FOVs"
):
    summary, issues = process_well_fov(
        patient,
        well_fov,
        expected_n_files_by_patient[patient],
        segmentation_masks_dir_by_patient[patient],
    )
    summaries.append(summary)
    all_issues.extend(issues)

summary_df = pd.DataFrame(summaries)
issues_df = pd.DataFrame(
    all_issues, columns=["patient", "well_fov", "issue_type", "detail"]
)
print(
    f"{len(summary_df)} well-FOVs processed across {len(PATIENTS)} patients, "
    f"{len(issues_df)} issues logged"
)


# ## Save the flattened per-well-FOV summary and the long-format issue log

Merging all patients' well-FOVs:   0%|          | 0/4187 [00:00<?, ?it/s]

<tifffile.TiffFile 'nuclei_mask.tiff'> contains no pages
<tifffile.TiffFile 'nuclei_mask.tiff'> contains no pages


4187 well-FOVs processed across 13 patients, 7298 issues logged


In [12]:
summary_out_path = logs_dir / "well_fov_feature_merge_summary_all_patients.csv"
issues_out_path = logs_dir / "well_fov_feature_merge_issues_all_patients.csv"

csv_summary_df = summary_df.copy()
for col in ["compartments_present", "compartment_row_counts"]:
    csv_summary_df[col] = csv_summary_df[col].apply(json.dumps)
csv_summary_df.to_csv(summary_out_path, index=False)
issues_df.to_csv(issues_out_path, index=False)

print(f"Wrote {summary_out_path}")
print(f"Wrote {issues_out_path}")


# ## Summarize findings, overall and per patient

Wrote /home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/3.cellprofiling/logs/well_fov_feature_merge_summary_all_patients.csv
Wrote /home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/3.cellprofiling/logs/well_fov_feature_merge_issues_all_patients.csv


In [13]:
n_well_fovs = len(summary_df)
n_no_files = int(summary_df["no_files_found"].sum())
n_file_count_mismatch = int(summary_df["file_count_mismatch"].sum())
n_with_issues = int((summary_df["n_issues"] > 0).sum())
n_misaligned = int(
    (summary_df["object_ids_aligned_across_compartments"] == False).sum()
)
n_misaligned_image_level = int(
    (summary_df["misalignment_root_cause"] == "image_level").sum()
)
n_misaligned_feature_level = int(
    (summary_df["misalignment_root_cause"] == "feature_extraction_level").sum()
)
n_misaligned_unknown = int(
    (summary_df["misalignment_root_cause"] == "unknown_mask_missing").sum()
)
n_read_errors = int(summary_df["n_read_errors"].sum())
n_merge_errors = int(summary_df["n_merge_errors"].sum())
n_merge_blowups = int(summary_df["n_merge_blowups"].sum())
n_organoid_profiles_empty = int(summary_df["organoid_profiles_empty"].sum())
n_organoid_object_ids_empty = int(summary_df["organoid_object_ids_empty"].sum())
n_organoid_empty_profile_merge_errors = int(
    summary_df["n_organoid_empty_profile_merge_errors"].sum()
)
n_source_mask_mismatches = int(summary_df["n_source_mask_mismatches"].sum())
n_source_mask_missing = int(summary_df["n_source_mask_missing"].sum())
n_well_fovs_with_source_mask_mismatch = int(
    (summary_df["n_source_mask_mismatches"] > 0).sum()
)
n_object_id_exceeds_threshold = int(summary_df["n_object_id_exceeds_threshold"].sum())
n_well_fovs_with_object_id_exceeds_threshold = int(
    (summary_df["n_object_id_exceeds_threshold"] > 0).sum()
)

issue_type_counts = (
    issues_df["issue_type"].value_counts() if len(issues_df) else pd.Series(dtype=int)
)

print(f"Well-FOVs processed:                {n_well_fovs}")
print(f"Well-FOVs with zero files found:    {n_no_files}")
print(f"Well-FOVs with >=1 issue:           {n_with_issues}")
print(f"Well-FOVs with file count mismatch: {n_file_count_mismatch}")
print(f"Well-FOVs with object-ID misalignment across compartments: {n_misaligned}")
print(f"  - traced to the source masks/images themselves: {n_misaligned_image_level}")
print(
    "  - traced to feature extraction/merging (masks agree, features don't): "
    f"{n_misaligned_feature_level}"
)
print(f"  - root cause unknown (a mask was missing): {n_misaligned_unknown}")
print(
    "Well-FOVs with empty Organoid profiles (informational only, NOT counted as an "
    f"issue): {n_organoid_profiles_empty}"
)
print(
    f"  - of which Organoid files present but zero object IDs: {n_organoid_object_ids_empty}"
)
print(
    "Organoid merges that failed due to an empty (0-row) profile file (a real merge "
    f"bug, reclassified from a generic merge error and still counted as an issue): "
    f"{n_organoid_empty_profile_merge_errors}"
)
print(
    "Well-FOVs where the segmentation mask's object IDs don't match the extracted "
    f"features (source likely stale/corrupted): {n_well_fovs_with_source_mask_mismatch}"
)
print(f"Total source-mask object-ID mismatches: {n_source_mask_mismatches}")
print(f"Total missing source mask files:   {n_source_mask_missing}")
print(
    "Well-FOVs with a profile whose object_id exceeds "
    f"{OBJECT_ID_SCHEME_THRESHOLD}: {n_well_fovs_with_object_id_exceeds_threshold}"
)
print(f"Total object_id-exceeds-threshold flags: {n_object_id_exceeds_threshold}")
print(f"Total read errors:  {n_read_errors}")
print(f"Total merge errors: {n_merge_errors}")
print(f"Total merge blow-ups: {n_merge_blowups}")
print()
print("Issue counts by type:")
issue_type_counts

Well-FOVs processed:                4187
Well-FOVs with zero files found:    47
Well-FOVs with >=1 issue:           1853
Well-FOVs with file count mismatch: 9
Well-FOVs with object-ID misalignment across compartments: 417
  - traced to the source masks/images themselves: 308
  - traced to feature extraction/merging (masks agree, features don't): 109
  - root cause unknown (a mask was missing): 0
Well-FOVs with empty Organoid profiles (informational only, NOT counted as an issue): 232
  - of which Organoid files present but zero object IDs: 0
Organoid merges that failed due to an empty (0-row) profile file (a real merge bug, reclassified from a generic merge error and still counted as an issue): 231
Well-FOVs where the segmentation mask's object IDs don't match the extracted features (source likely stale/corrupted): 164
Total source-mask object-ID mismatches: 301
Total missing source mask files:   0
Well-FOVs with a profile whose object_id exceeds 255: 1272
Total object_id-exceeds-thres

issue_type
object_id_exceeds_threshold                  5056
object_id_misalignment                        645
object_id_misalignment_traced_to_image        416
merge_error                                   362
source_mask_object_id_mismatch                301
organoid_empty_profile_merge_error            231
missing_merge_keys                            120
object_id_misalignment_traced_to_features     109
no_files_found                                 47
file_count_mismatch                             9
empty_organoid_profiles                         1
read_error                                      1
Name: count, dtype: int64

In [14]:
per_patient_summary = (
    summary_df.groupby("patient")
    .agg(
        n_well_fovs=("well_fov", "count"),
        n_no_files=("no_files_found", "sum"),
        n_file_count_mismatch=("file_count_mismatch", "sum"),
        n_with_issues=("n_issues", lambda s: (s > 0).sum()),
        n_organoid_profiles_empty=("organoid_profiles_empty", "sum"),
        n_organoid_empty_profile_merge_errors=(
            "n_organoid_empty_profile_merge_errors",
            "sum",
        ),
        n_source_mask_mismatches=("n_source_mask_mismatches", "sum"),
        n_source_mask_missing=("n_source_mask_missing", "sum"),
        n_object_id_exceeds_threshold=("n_object_id_exceeds_threshold", "sum"),
    )
    .reset_index()
)
per_patient_summary["expected_n_files"] = per_patient_summary["patient"].map(
    expected_n_files_by_patient
)
per_patient_summary

,patient,n_well_fovs,n_no_files,n_file_count_mismatch,n_with_issues,n_organoid_profiles_empty,n_organoid_empty_profile_merge_errors,n_source_mask_mismatches,n_source_mask_missing,n_object_id_exceeds_threshold,expected_n_files
0,NF0014_T1,102,0,0,66,2,2,0,0,260,101
1,NF0014_T2,350,0,0,123,16,16,0,0,396,101
2,NF0016_T1,169,47,1,87,7,7,8,0,312,101
3,NF0018_T6,160,0,0,107,11,11,18,0,370,101
4,NF0021_T1,348,0,0,190,7,7,10,0,744,101
5,NF0030_T1,207,0,3,169,7,7,8,0,656,101
6,NF0035_T1,349,0,1,56,6,5,13,0,156,101
7,NF0037_T1,420,0,0,76,6,6,50,0,0,101
8,NF0037_T1_CQ1,693,0,0,292,80,80,78,0,236,101
9,NF0040_T1,420,0,2,173,22,22,2,0,480,101


In [15]:
summary_df.loc[
    summary_df["object_ids_aligned_across_compartments"] == False,
    [
        "patient",
        "well_fov",
        "misalignment_root_cause",
        "compartments_present",
        "compartment_row_counts",
    ],
]


# ## Root-causing the object-ID misalignment: image vs. feature extraction
#
# For every well-FOV flagged with `object_id_misalignment` above, the Nuclei/Cell/
# Cytoplasm/Nucleocentric segmentation mask TIFFs implicated in that misalignment were
# read directly and compared to **each other** (not just to their own extracted
# features, as in the source-mask consistency check below). This distinguishes two
# root causes that look identical from the merged feature tables alone:
# - `image_level`: the masks themselves already disagree on object IDs — the
#   misalignment originates upstream, in segmentation, not in this pipeline's
#   feature extraction or merge code.
# - `feature_extraction_level`: the masks agree with each other, but the extracted
#   feature tables for those compartments don't — the misalignment was introduced
#   while extracting or writing out `object_id` per compartment.
# - `unknown_mask_missing`: at least one implicated compartment's mask file was
#   missing on disk, so the comparison couldn't be made.

,patient,well_fov,misalignment_root_cause,compartments_present,compartment_row_counts
115,NF0014_T2,C11-7,image_level,"[Cell, Cytoplasm, Nuclei, Nucleocentric, Organ...","{'Organoid': 1, 'Nuclei': 5, 'Cell': 5, 'Cytop..."
142,NF0014_T2,C5-6,image_level,"[Cell, Cytoplasm, Nuclei, Nucleocentric, Organ...","{'Organoid': 3, 'Nuclei': 7, 'Cell': 7, 'Cytop..."
148,NF0014_T2,C6-5,image_level,"[Cell, Cytoplasm, Nuclei, Nucleocentric, Organ...","{'Organoid': 5, 'Nuclei': 25, 'Cell': 25, 'Cyt..."
154,NF0014_T2,C7-4,image_level,"[Cell, Cytoplasm, Nuclei, Nucleocentric, Organ...","{'Organoid': 3, 'Nuclei': 5, 'Cell': 5, 'Cytop..."
170,NF0014_T2,C9-6,image_level,"[Cell, Cytoplasm, Nuclei, Nucleocentric, Organ...","{'Organoid': 8, 'Nuclei': 13, 'Cell': 13, 'Cyt..."
...,...,...,...,...,...
3902,SARCO361_T1,C9-3,image_level,"[Cell, Cytoplasm, Nuclei, Nucleocentric, Organ...","{'Organoid': 1, 'Nuclei': 8, 'Cell': 8, 'Cytop..."
3910,SARCO361_T1,D10-4,image_level,"[Cell, Cytoplasm, Nuclei, Nucleocentric, Organ...","{'Organoid': 8, 'Nuclei': 15, 'Cell': 15, 'Cyt..."
3921,SARCO361_T1,D2-1,image_level,"[Cell, Cytoplasm, Nuclei, Nucleocentric, Organ...","{'Organoid': 12, 'Nuclei': 19, 'Cell': 19, 'Cy..."
3949,SARCO361_T1,D6-1,image_level,"[Cell, Cytoplasm, Nuclei, Nucleocentric, Organ...","{'Organoid': 13, 'Nuclei': 25, 'Cell': 25, 'Cy..."


In [16]:
issues_df.loc[
    issues_df["issue_type"].isin(
        [
            "object_id_misalignment_traced_to_image",
            "object_id_misalignment_traced_to_features",
        ]
    )
].sort_values(["patient", "well_fov"]).reset_index(drop=True)

,patient,well_fov,issue_type,detail
0,NF0014_T2,C11-7,object_id_misalignment_traced_to_image,Cytoplasm vs Nuclei: the segmentation masks th...
1,NF0014_T2,C5-6,object_id_misalignment_traced_to_image,Cytoplasm vs Nuclei: the segmentation masks th...
2,NF0014_T2,C6-5,object_id_misalignment_traced_to_image,Cytoplasm vs Nuclei: the segmentation masks th...
3,NF0014_T2,C7-4,object_id_misalignment_traced_to_image,Cytoplasm vs Nuclei: the segmentation masks th...
4,NF0014_T2,C9-6,object_id_misalignment_traced_to_image,Cytoplasm vs Nuclei: the segmentation masks th...
...,...,...,...,...
520,SARCO361_T1,C9-3,object_id_misalignment_traced_to_image,Cytoplasm vs Nuclei: the segmentation masks th...
521,SARCO361_T1,D10-4,object_id_misalignment_traced_to_image,Cytoplasm vs Nuclei: the segmentation masks th...
522,SARCO361_T1,D2-1,object_id_misalignment_traced_to_image,Cytoplasm vs Nuclei: the segmentation masks th...
523,SARCO361_T1,D6-1,object_id_misalignment_traced_to_image,Cytoplasm vs Nuclei: the segmentation masks th...


In [17]:
misalignment_root_cause_counts = (
    summary_df.loc[
        summary_df["object_ids_aligned_across_compartments"] == False,
        "misalignment_root_cause",
    ]
    .value_counts(dropna=False)
    .rename_axis("misalignment_root_cause")
    .reset_index(name="n_well_fovs")
)
misalignment_root_cause_counts


# ## Well-FOVs/compartments with an object_id exceeding 255
#
# Small sequential IDs (1, 2, 3, ...) are the expected "sequential" scheme. A compartment
# containing any `object_id` greater than 255 is likely using the "z_slice_global" scheme
# instead — worth a closer look, and one of the triggers for the source-mask consistency
# check below.

,misalignment_root_cause,n_well_fovs
0,image_level,308
1,feature_extraction_level,109


In [18]:
issues_df.loc[issues_df["issue_type"] == "object_id_exceeds_threshold"].sort_values(
    ["patient", "well_fov"]
).reset_index(drop=True)


# ## Organoid profile checks (informational, not counted as issues)
#
# Well-FOVs with no readable Organoid feature files at all, or an Organoid merge that
# produced zero rows. Some well-FOVs legitimately have no organoids, so these are tracked
# for visibility only and do not count toward `n_issues` or trigger the source-mask
# consistency check on their own. Also shown: Organoid merges that failed outright
# because one of their input files was an empty (0-row) profile — a real merge bug
# (dtype-clash symptom, reclassified from a generic `merge_error`), which **is** still
# counted as an issue.

,patient,well_fov,issue_type,detail
0,NF0014_T1,C10-1,object_id_exceeds_threshold,Nuclei: object_id values across 24 input file(...
1,NF0014_T1,C10-1,object_id_exceeds_threshold,Cell: object_id values across 23 input file(s)...
2,NF0014_T1,C10-1,object_id_exceeds_threshold,Cytoplasm: object_id values across 23 input fi...
3,NF0014_T1,C10-1,object_id_exceeds_threshold,Nucleocentric: object_id values across 8 input...
4,NF0014_T1,C10-2,object_id_exceeds_threshold,Nuclei: object_id values across 24 input file(...
...,...,...,...,...
5051,SARCO361_T1,G9-5,object_id_exceeds_threshold,Nucleocentric: object_id values across 8 input...
5052,SARCO361_T1,G9-7,object_id_exceeds_threshold,Nuclei: object_id values across 24 input file(...
5053,SARCO361_T1,G9-7,object_id_exceeds_threshold,Cell: object_id values across 23 input file(s)...
5054,SARCO361_T1,G9-7,object_id_exceeds_threshold,Cytoplasm: object_id values across 23 input fi...


In [19]:
issues_df.loc[
    issues_df["issue_type"].isin(
        [
            "empty_organoid_profiles",
            "organoid_object_ids_empty",
            "organoid_empty_profile_merge_error",
        ]
    )
].sort_values(["patient", "well_fov"]).reset_index(drop=True)


# ## Source mask consistency check
#
# For every well-FOV/compartment implicated in a merge-related issue (blow-up,
# duplicate `object_id`, a failed Organoid merge, an object ID over 255, or
# cross-compartment misalignment), the segmentation mask TIFF was read directly and its
# unique object IDs compared against the object IDs present in the extracted feature
# files. A mismatch here means the mask on disk itself doesn't match what was extracted
# — most likely it was re-segmented or overwritten after featurization last ran — as
# opposed to a bug in the merge/extraction code.

,patient,well_fov,issue_type,detail
0,NF0014_T1,D2-2,organoid_empty_profile_merge_error,Organoid merge failed because ['Organoid_AGP_G...
1,NF0014_T1,E2-2,organoid_empty_profile_merge_error,Organoid merge failed because ['Organoid_AGP_G...
2,NF0014_T2,C3-6,organoid_empty_profile_merge_error,Organoid merge failed because ['Organoid_AGP_G...
3,NF0014_T2,C4-5,organoid_empty_profile_merge_error,Organoid merge failed because ['Organoid_AGP_G...
4,NF0014_T2,D10-1,organoid_empty_profile_merge_error,Organoid merge failed because ['Organoid_AGP_G...
...,...,...,...,...
227,SARCO219_T2,C4-2,organoid_empty_profile_merge_error,Organoid merge failed because ['Organoid_AGP_G...
228,SARCO219_T2,E11-3,organoid_empty_profile_merge_error,Organoid merge failed because ['Organoid_AGP_G...
229,SARCO219_T2,G10-2,organoid_empty_profile_merge_error,Organoid merge failed because ['Organoid_AGP_G...
230,SARCO361_T1,C2-1,organoid_empty_profile_merge_error,Organoid merge failed because ['Organoid_AGP_G...


In [20]:
issues_df.loc[
    issues_df["issue_type"].isin(
        ["source_mask_object_id_mismatch", "source_mask_missing"]
    )
].sort_values(["patient", "well_fov"]).reset_index(drop=True)


# ## Generate the markdown report

,patient,well_fov,issue_type,detail
0,NF0016_T1,C10-1,source_mask_object_id_mismatch,"Cell: mask has 19 object IDs, extracted featur..."
1,NF0016_T1,C10-1,source_mask_object_id_mismatch,"Cytoplasm: mask has 18 object IDs, extracted f..."
2,NF0016_T1,C10-1,source_mask_object_id_mismatch,"Nuclei: mask has 19 object IDs, extracted feat..."
3,NF0016_T1,C10-1,source_mask_object_id_mismatch,"Nucleocentric: mask has 19 object IDs, extract..."
4,NF0016_T1,E4-1,source_mask_object_id_mismatch,"Cell: mask has 13 object IDs, extracted featur..."
...,...,...,...,...
296,SARCO361_T1,D6-1,source_mask_object_id_mismatch,"Nucleocentric: mask has 25 object IDs, extract..."
297,SARCO361_T1,E10-2,source_mask_object_id_mismatch,"Cell: mask has 8 object IDs, extracted feature..."
298,SARCO361_T1,E10-2,source_mask_object_id_mismatch,"Cytoplasm: mask has 7 object IDs, extracted fe..."
299,SARCO361_T1,E10-2,source_mask_object_id_mismatch,"Nuclei: mask has 8 object IDs, extracted featu..."


In [21]:
worst_offenders = summary_df.sort_values("n_issues", ascending=False).loc[
    summary_df["n_issues"] > 0,
    [
        "patient",
        "well_fov",
        "n_issues",
        "file_count_mismatch",
        "object_ids_aligned_across_compartments",
        "misalignment_root_cause",
        "organoid_profiles_empty",
        "n_source_mask_mismatches",
        "n_object_id_exceeds_threshold",
    ],
]

misalignment_root_cause_issues_df = issues_df.loc[
    issues_df["issue_type"].isin(
        [
            "object_id_misalignment_traced_to_image",
            "object_id_misalignment_traced_to_features",
        ]
    )
].sort_values(["patient", "well_fov"])
misalignment_root_cause_section_lines = (
    misalignment_root_cause_issues_df.to_markdown(index=False)
    if len(misalignment_root_cause_issues_df)
    else "_None._"
)
misalignment_root_cause_counts_lines = (
    misalignment_root_cause_counts.to_markdown(index=False)
    if len(misalignment_root_cause_counts)
    else "_None._"
)

organoid_issues_df = issues_df.loc[
    issues_df["issue_type"].isin(
        [
            "empty_organoid_profiles",
            "organoid_object_ids_empty",
            "organoid_empty_profile_merge_error",
        ]
    )
].sort_values(["patient", "well_fov"])
organoid_section_lines = (
    organoid_issues_df.to_markdown(index=False)
    if len(organoid_issues_df)
    else "_None._"
)

source_mask_issues_df = issues_df.loc[
    issues_df["issue_type"].isin(
        ["source_mask_object_id_mismatch", "source_mask_missing"]
    )
].sort_values(["patient", "well_fov"])
source_mask_section_lines = (
    source_mask_issues_df.to_markdown(index=False)
    if len(source_mask_issues_df)
    else "_None._"
)

object_id_threshold_issues_df = issues_df.loc[
    issues_df["issue_type"] == "object_id_exceeds_threshold"
].sort_values(["patient", "well_fov"])
object_id_threshold_section_lines = (
    object_id_threshold_issues_df.to_markdown(index=False)
    if len(object_id_threshold_issues_df)
    else "_None._"
)

report_lines = [
    "# Feature Merge QC Report — All Patients",
    "",
    f"Generated across {len(PATIENTS)} patients from "
    f"`{profile_base_dir}/data/{{patient}}/{output_features_subparent_name}`.",
    "",
    "## Summary",
    "",
    f"- Well-FOVs processed: **{n_well_fovs}**",
    f"- Well-FOVs with zero files found: **{n_no_files}**",
    f"- Well-FOVs with at least one issue: **{n_with_issues}**",
    f"- Well-FOVs with a file-count mismatch (vs. that patient's own expected count): "
    f"**{n_file_count_mismatch}**",
    f"- Well-FOVs with object-ID misalignment across compartments: **{n_misaligned}**",
    f"  - traced to the source masks/images themselves: **{n_misaligned_image_level}**",
    "  - traced to feature extraction/merging (masks agree, features don't): "
    f"**{n_misaligned_feature_level}**",
    f"  - root cause unknown (a mask was missing): **{n_misaligned_unknown}**",
    f"- Well-FOVs with empty Organoid profiles (*informational, not counted as an issue*): **{n_organoid_profiles_empty}**",
    f"- Well-FOVs with Organoid files present but zero object IDs (*informational, not counted as an issue*): **{n_organoid_object_ids_empty}**",
    f"- Organoid merges that failed due to an empty (0-row) profile file (a real merge bug, reclassified from a generic merge error — still counted as an issue): **{n_organoid_empty_profile_merge_errors}**",
    f"- Well-FOVs where the source segmentation mask's object IDs don't match the "
    f"extracted features: **{n_well_fovs_with_source_mask_mismatch}**",
    f"- Total source-mask object-ID mismatches: **{n_source_mask_mismatches}**",
    f"- Total missing source mask files: **{n_source_mask_missing}**",
    f"- Well-FOVs with a profile whose object_id exceeds {OBJECT_ID_SCHEME_THRESHOLD}: **{n_well_fovs_with_object_id_exceeds_threshold}**",
    f"- Total object_id-exceeds-threshold flags: **{n_object_id_exceeds_threshold}**",
    f"- Total read errors: **{n_read_errors}**",
    f"- Total merge errors: **{n_merge_errors}**",
    f"- Total merge blow-ups: **{n_merge_blowups}**",
    "",
    "## Per-patient breakdown",
    per_patient_summary.to_markdown(index=False),
    "",
    "## Issue counts by type",
    "",
    issue_type_counts.to_frame("count").to_markdown()
    if len(issue_type_counts)
    else "_No issues found._",
    "",
    "## Well-FOVs with zero files found",
    "",
    summary_df.loc[summary_df["no_files_found"], ["patient", "well_fov"]].to_markdown(
        index=False
    )
    if n_no_files
    else "_None._",
    "",
    "## Organoid profile checks (informational, not counted as issues)",
    "",
    "Well-FOVs with no readable Organoid feature files at all, or an Organoid merge ",
    "that produced zero rows (tracked for visibility only, not counted toward ",
    "n_issues). Also shown: Organoid merges that failed outright because one input ",
    "file was an empty (0-row) profile — a real merge bug, and still counted as an ",
    "issue.",
    "",
    organoid_section_lines,
    "",
    "## Object IDs exceeding {}".format(OBJECT_ID_SCHEME_THRESHOLD),
    "",
    "Compartments containing any object_id greater than "
    f"{OBJECT_ID_SCHEME_THRESHOLD} — likely the 'z_slice_global' ID scheme rather ",
    "than small sequential IDs for that well-FOV/compartment.",
    "",
    object_id_threshold_section_lines,
    "",
    "## Root-causing the object-ID misalignment: image vs. feature extraction",
    "",
    "For every well-FOV flagged with `object_id_misalignment`, the Nuclei/Cell/",
    "Cytoplasm/Nucleocentric masks implicated in that misalignment were read directly ",
    "and compared to **each other** — not just to their own extracted features (that's ",
    "the source-mask consistency check below). `image_level` means the masks ",
    "themselves already disagree, so the root cause is upstream in segmentation, not ",
    "in this pipeline. `feature_extraction_level` means the masks agree with each ",
    "other but the extracted feature tables don't, so the root cause is in how ",
    "`object_id` was extracted/written per compartment. `unknown_mask_missing` means a ",
    "mask file needed for the comparison wasn't found on disk.",
    "",
    misalignment_root_cause_counts_lines,
    "",
    misalignment_root_cause_section_lines,
    "",
    "## Source mask consistency check",
    "",
    "For every well-FOV/compartment implicated in a merge-related issue (blow-up, ",
    "duplicate object_id, a failed Organoid merge, an object ID over ",
    f"{OBJECT_ID_SCHEME_THRESHOLD}, or cross-compartment misalignment), the ",
    "segmentation mask TIFF was read directly and its unique object IDs compared ",
    "against the object IDs in the extracted features. A mismatch points to the mask ",
    "on disk (likely re-segmented/overwritten after featurization last ran) rather ",
    "than a merge bug.",
    "",
    source_mask_section_lines,
    "",
    "## Well-FOVs with the most issues (top 30)",
    "",
    worst_offenders.head(30).to_markdown(index=False)
    if len(worst_offenders)
    else "_None._",
    "",
    "## Notes",
    "",
    "- Full per-issue detail is in `well_fov_feature_merge_issues_all_patients.csv`.",
    "- An empty Organoid profile (no files, or a merge with zero rows) is tracked",
    "  separately and does NOT count as an issue — some well-FOVs legitimately have",
    "  no organoids. An Organoid merge that fails outright because one input file is",
    "  an empty (0-row) profile IS still counted, since that reflects an actual",
    "  merge bug rather than a legitimately-empty well-FOV.",
    "- The source mask consistency check reads segmentation mask TIFFs directly (not ",
    "  through the extracted features) to determine whether an object-ID discrepancy ",
    "  originates from the mask itself vs. from feature extraction/merging.",
    "- The root-cause check goes one step further for `object_id_misalignment` cases",
    "  specifically: it compares the implicated compartments' masks directly against",
    "  EACH OTHER (not just each mask against its own features), which is the only way",
    "  to tell an image-level misalignment (masks disagree with each other) apart from",
    "  a feature-extraction-level one (masks agree, but the feature tables don't).",
    "- `Organoid` is excluded from the cross-compartment object-ID alignment check; it's",
    "  a distinct feature space (one row per whole organoid) with its own object-ID",
    "  space that isn't expected to match the single-cell compartments.",
    "- Each patient's expected file count is that patient's own mode file count across",
    "  its well-FOVs (excluding zero-file well-FOVs) — patients can have different",
    "  channel/feature-type combinations, so a single global expected count doesn't",
    "  generalize across patients.",
]

report_text = "\n".join(report_lines)
report_out_path = logs_dir / "well_fov_feature_merge_report_all_patients.md"
report_out_path.write_text(report_text)
print(f"Wrote {report_out_path}")

Wrote /home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/3.cellprofiling/logs/well_fov_feature_merge_report_all_patients.md


In [22]:
from IPython.display import Markdown, display

display(Markdown(report_text))

# Feature Merge QC Report — All Patients

Generated across 13 patients from `/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/{patient}/extracted_features`.

## Summary

- Well-FOVs processed: **4187**
- Well-FOVs with zero files found: **47**
- Well-FOVs with at least one issue: **1853**
- Well-FOVs with a file-count mismatch (vs. that patient's own expected count): **9**
- Well-FOVs with object-ID misalignment across compartments: **417**
  - traced to the source masks/images themselves: **308**
  - traced to feature extraction/merging (masks agree, features don't): **109**
  - root cause unknown (a mask was missing): **0**
- Well-FOVs with empty Organoid profiles (*informational, not counted as an issue*): **232**
- Well-FOVs with Organoid files present but zero object IDs (*informational, not counted as an issue*): **0**
- Organoid merges that failed due to an empty (0-row) profile file (a real merge bug, reclassified from a generic merge error — still counted as an issue): **231**
- Well-FOVs where the source segmentation mask's object IDs don't match the extracted features: **164**
- Total source-mask object-ID mismatches: **301**
- Total missing source mask files: **0**
- Well-FOVs with a profile whose object_id exceeds 255: **1272**
- Total object_id-exceeds-threshold flags: **5056**
- Total read errors: **1**
- Total merge errors: **362**
- Total merge blow-ups: **0**

## Per-patient breakdown
| patient       |   n_well_fovs |   n_no_files |   n_file_count_mismatch |   n_with_issues |   n_organoid_profiles_empty |   n_organoid_empty_profile_merge_errors |   n_source_mask_mismatches |   n_source_mask_missing |   n_object_id_exceeds_threshold |   expected_n_files |
|:--------------|--------------:|-------------:|------------------------:|----------------:|----------------------------:|----------------------------------------:|---------------------------:|------------------------:|--------------------------------:|-------------------:|
| NF0014_T1     |           102 |            0 |                       0 |              66 |                           2 |                                       2 |                          0 |                       0 |                             260 |                101 |
| NF0014_T2     |           350 |            0 |                       0 |             123 |                          16 |                                      16 |                          0 |                       0 |                             396 |                101 |
| NF0016_T1     |           169 |           47 |                       1 |              87 |                           7 |                                       7 |                          8 |                       0 |                             312 |                101 |
| NF0018_T6     |           160 |            0 |                       0 |             107 |                          11 |                                      11 |                         18 |                       0 |                             370 |                101 |
| NF0021_T1     |           348 |            0 |                       0 |             190 |                           7 |                                       7 |                         10 |                       0 |                             744 |                101 |
| NF0030_T1     |           207 |            0 |                       3 |             169 |                           7 |                                       7 |                          8 |                       0 |                             656 |                101 |
| NF0035_T1     |           349 |            0 |                       1 |              56 |                           6 |                                       5 |                         13 |                       0 |                             156 |                101 |
| NF0037_T1     |           420 |            0 |                       0 |              76 |                           6 |                                       6 |                         50 |                       0 |                               0 |                101 |
| NF0037_T1_CQ1 |           693 |            0 |                       0 |             292 |                          80 |                                      80 |                         78 |                       0 |                             236 |                101 |
| NF0040_T1     |           420 |            0 |                       2 |             173 |                          22 |                                      22 |                          2 |                       0 |                             480 |                101 |
| NF0055_T1     |           420 |            0 |                       1 |             131 |                          62 |                                      62 |                          4 |                       0 |                               0 |                101 |
| SARCO219_T2   |           199 |            0 |                       0 |             135 |                           4 |                                       4 |                         86 |                       0 |                             460 |                101 |
| SARCO361_T1   |           350 |            0 |                       1 |             248 |                           2 |                                       2 |                         24 |                       0 |                             986 |                101 |

## Issue counts by type

| issue_type                                |   count |
|:------------------------------------------|--------:|
| object_id_exceeds_threshold               |    5056 |
| object_id_misalignment                    |     645 |
| object_id_misalignment_traced_to_image    |     416 |
| merge_error                               |     362 |
| source_mask_object_id_mismatch            |     301 |
| organoid_empty_profile_merge_error        |     231 |
| missing_merge_keys                        |     120 |
| object_id_misalignment_traced_to_features |     109 |
| no_files_found                            |      47 |
| file_count_mismatch                       |       9 |
| empty_organoid_profiles                   |       1 |
| read_error                                |       1 |

## Well-FOVs with zero files found

| patient   | well_fov   |
|:----------|:-----------|
| NF0016_T1 | C10-3      |
| NF0016_T1 | C11-1      |
| NF0016_T1 | C11-2      |
| NF0016_T1 | C2-1       |
| NF0016_T1 | C2-2       |
| NF0016_T1 | C3-3       |
| NF0016_T1 | C7-3       |
| NF0016_T1 | C8-3       |
| NF0016_T1 | C8-4       |
| NF0016_T1 | C8-5       |
| NF0016_T1 | C9-3       |
| NF0016_T1 | D10-3      |
| NF0016_T1 | D11-3      |
| NF0016_T1 | D11-4      |
| NF0016_T1 | D2-4       |
| NF0016_T1 | D3-2       |
| NF0016_T1 | D3-3       |
| NF0016_T1 | D4-3       |
| NF0016_T1 | D6-3       |
| NF0016_T1 | D7-3       |
| NF0016_T1 | E-3        |
| NF0016_T1 | E2-3       |
| NF0016_T1 | E3-3       |
| NF0016_T1 | E3-4       |
| NF0016_T1 | E7-3       |
| NF0016_T1 | E7-4       |
| NF0016_T1 | E8-3       |
| NF0016_T1 | E8-5       |
| NF0016_T1 | E8-6       |
| NF0016_T1 | F10-3      |
| NF0016_T1 | F2-3       |
| NF0016_T1 | F4-4       |
| NF0016_T1 | F6-3       |
| NF0016_T1 | F8-4       |
| NF0016_T1 | F9-4       |
| NF0016_T1 | G11-1      |
| NF0016_T1 | G2-1       |
| NF0016_T1 | G2-2       |
| NF0016_T1 | G3-2       |
| NF0016_T1 | G3-3       |
| NF0016_T1 | G4-4       |
| NF0016_T1 | G6-4       |
| NF0016_T1 | G8-4       |
| NF0016_T1 | G8-5       |
| NF0016_T1 | G9-3       |
| NF0016_T1 | G9-4       |
| NF0016_T1 | G9-5       |

## Organoid profile checks (informational, not counted as issues)

Well-FOVs with no readable Organoid feature files at all, or an Organoid merge 
that produced zero rows (tracked for visibility only, not counted toward 
n_issues). Also shown: Organoid merges that failed outright because one input 
file was an empty (0-row) profile — a real merge bug, and still counted as an 
issue.

| patient       | well_fov   | issue_type                         | detail                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
|:--------------|:-----------|:-----------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| NF0014_T1     | D2-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T1     | E2-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | C3-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | C4-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | D10-1      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | E2-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | E2-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | E4-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | F2-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | F4-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | G10-3      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | G11-4      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | G2-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | G2-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | G5-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | G7-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | G7-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0014_T2     | G9-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0016_T1     | D2-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0016_T1     | D2-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0016_T1     | D5-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0016_T1     | E3-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0016_T1     | F4-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0016_T1     | F7-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0016_T1     | F9-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | C10-2      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | C8-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | C8-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | D2-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | D3-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | D5-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | D9-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | E7-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | E9-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | F11-1      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0018_T6     | G9-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0021_T1     | C2-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0021_T1     | C2-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0021_T1     | C3-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0021_T1     | D7-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0021_T1     | E2-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0021_T1     | E3-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0021_T1     | E5-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0030_T1     | D2-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0030_T1     | D6-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0030_T1     | F10-1      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0030_T1     | F6-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0030_T1     | G10-2      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0030_T1     | G4-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0030_T1     | G5-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0035_T1     | C9-7       | empty_organoid_profiles            | no readable Organoid feature files for C9-7; Organoid merge skipped                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          |
| NF0035_T1     | D9-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0035_T1     | F5-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0035_T1     | F5-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0035_T1     | F9-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0035_T1     | G2-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1     | B3-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1     | D6-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1     | E11-4      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1     | F10-3      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1     | F10-7      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1     | G3-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B10-1      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B11-3      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B2-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B2-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B3-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B3-19      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B3-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B3-21      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B3-8       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B3-9       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | B7-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C10-2      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C10-7      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C2-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C3-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C6-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C7-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C8-12      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C8-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C8-9       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C9-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C9-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | C9-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D10-13     | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D10-6      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D11-1      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D11-12     | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D2-18      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D2-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D2-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D2-8       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D7-12      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D7-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | D8-8       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E2-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E2-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E5-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E5-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E5-8       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E5-9       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E6-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E7-10      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E7-11      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E7-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E7-8       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E7-9       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E8-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | E9-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F10-13     | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F10-14     | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F10-16     | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F10-26     | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F10-27     | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F10-30     | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F10-7      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F11-9      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F2-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F2-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F3-10      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F3-11      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F3-14      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F3-8       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F3-9       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F4-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F8-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F8-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F8-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | F8-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G11-1      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G11-5      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G2-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G2-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G2-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G3-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G3-9       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G6-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G7-11      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G7-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G8-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0037_T1_CQ1 | G8-9       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | B11-1      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | B2-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | B7-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | C11-5      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | C11-6      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | C2-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | C2-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | C3-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | C8-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | D4-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | D5-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | D5-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | D7-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | D9-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | E10-4      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | E11-5      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | E2-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | F3-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | F4-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | F5-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | G2-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0040_T1     | G6-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | B2-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | B4-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | B4-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | B4-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | B4-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | B4-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | B4-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C3-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C3-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C3-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C3-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C3-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C3-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C4-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C4-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C4-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C4-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C4-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | C6-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D5-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D5-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D5-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D5-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D5-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D6-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D6-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D6-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D6-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D6-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D7-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D7-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D7-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D7-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D7-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | D7-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E4-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E5-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E5-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E5-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E5-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E6-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E6-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E6-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E6-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E8-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | E8-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | F3-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | F3-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | F3-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | F3-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | F3-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | F6-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | F6-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | F6-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | F6-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | G11-4      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | G3-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | G3-3       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | G3-4       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | G3-5       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | G3-6       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| NF0055_T1     | G3-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| SARCO219_T2   | C10-3      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| SARCO219_T2   | C4-2       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| SARCO219_T2   | E11-3      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| SARCO219_T2   | G10-2      | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| SARCO361_T1   | C2-1       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |
| SARCO361_T1   | E4-7       | organoid_empty_profile_merge_error | Organoid merge failed because ['Organoid_AGP_Granularity_CPU_features.parquet', 'Organoid_DNA_SAMMed3D_GPU_features.parquet', 'Organoid_NoChannel_AreaSizeShape_CPU_features.parquet', 'Organoid_DNA-ER_Colocalization_CPU_features.parquet', 'Organoid_DNA_Texture_CPU_features.parquet', 'Organoid_Mito_Intensity_CPU_features.parquet', 'Organoid_DNA-Mito_Colocalization_CPU_features.parquet', 'Organoid_AGP_SAMMed3D_GPU_features.parquet', 'Organoid_Mito_Granularity_CPU_features.parquet', 'Organoid_Mito_SAMMed3D_GPU_features.parquet', 'Organoid_ER-Mito_Colocalization_CPU_features.parquet', 'Organoid_ER_Texture_CPU_features.parquet', 'Organoid_DNA_Granularity_CPU_features.parquet', 'Organoid_AGP_Texture_CPU_features.parquet', 'Organoid_AGP_Intensity_CPU_features.parquet', 'Organoid_ER_Granularity_CPU_features.parquet', 'Organoid_Mito-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_SAMMed3D_GPU_features.parquet', 'Organoid_ER-AGP_Colocalization_CPU_features.parquet', 'Organoid_ER_Intensity_CPU_features.parquet', 'Organoid_DNA-AGP_Colocalization_CPU_features.parquet', 'Organoid_Mito_Texture_CPU_features.parquet', 'Organoid_DNA_Intensity_CPU_features.parquet'] contain 0 rows (an empty object_id column typed as float64 conflicts with the object/int dtype in the populated Organoid files during merge). Original error: You are trying to merge on float64 and object columns for key 'object_id'. If you wish to proceed you should use pd.concat |

## Object IDs exceeding 255

Compartments containing any object_id greater than 255 — likely the 'z_slice_global' ID scheme rather 
than small sequential IDs for that well-FOV/compartment.

| patient       | well_fov   | issue_type                  | detail                                                                                                                                                                                        |
|:--------------|:-----------|:----------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| NF0014_T1     | C10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0014_T1     | C4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0014_T1     | C4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0014_T1     | C4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0014_T1     | C4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 11565, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0014_T1     | C4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11565, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0014_T1     | C4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11565, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0014_T1     | C4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 11565, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0014_T1     | C5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | C9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | C9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | C9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | C9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | D8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | D8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | D8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | D8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0014_T1     | E4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0014_T1     | E4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0014_T1     | E4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0014_T1     | E4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | E9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | E9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | E9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | E9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | F10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | F10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | F10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | F10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | F11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | F11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | F11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | F11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | F2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | F2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | F2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | F2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | F2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | F2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | F2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | F2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | F3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | F3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | F3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | F3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | F4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | F4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | F4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | F4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | F5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | F5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | F5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | F5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | F7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | F7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | F7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | F7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | F7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0014_T1     | F7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0014_T1     | F7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0014_T1     | F7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0014_T1     | F9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | F9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | F9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | F9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0014_T1     | G6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0014_T1     | G6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0014_T1     | G6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0014_T1     | G7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T1     | G8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T1     | G8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T1     | G8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T1     | G9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0014_T1     | G9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0014_T1     | G9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0014_T1     | G9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0014_T2     | C10-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C10-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C10-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C10-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | C9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | C9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | C9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | C9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | D8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | D8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | D8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | D8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 14392, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0014_T2     | E11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 14392, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0014_T2     | E11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 14392, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0014_T2     | E11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 14392, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0014_T2     | E2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0014_T2     | E6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0014_T2     | E6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0014_T2     | E6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0014_T2     | E7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E8-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E8-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E8-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E8-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | E9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | E9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | E9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | E9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F10-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F10-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F10-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F10-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0014_T2     | F3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0014_T2     | F3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0014_T2     | F3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0014_T2     | F3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F5-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F5-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F5-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F5-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | F9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | F9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | F9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | F9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G7-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G7-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G7-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G7-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0014_T2     | G8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0014_T2     | G8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0014_T2     | G8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0014_T2     | G8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0016_T1     | C4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0016_T1     | C4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0016_T1     | C4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0016_T1     | C4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0016_T1     | C4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0016_T1     | C4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0016_T1     | C4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0016_T1     | C4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | C9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | C9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | C9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | C9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | D9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | D9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | D9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | D9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | E9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | E9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | E9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | E9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0016_T1     | F8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0016_T1     | F8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0016_T1     | F8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0016_T1     | F8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0016_T1     | F8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0016_T1     | F8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0016_T1     | F8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0016_T1     | F9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | F9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | F9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | F9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | F9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0016_T1     | G6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0016_T1     | G6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0016_T1     | G6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0016_T1     | G7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0016_T1     | G9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0016_T1     | G9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0016_T1     | G9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0016_T1     | G9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | C11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0018_T6     | C11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0018_T6     | C11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0018_T6     | C3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | C6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0018_T6     | C6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0018_T6     | C6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0018_T6     | C6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | C6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0018_T6     | C6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0018_T6     | C6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0018_T6     | C7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 12336, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0018_T6     | C8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 12336, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | C8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 12336, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0018_T6     | C8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 12336, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0018_T6     | C8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | C9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | C9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | C9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | C9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | D9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | D9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | D9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | D9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E-3        | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 12593, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0018_T6     | E11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 12593, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | E11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 12593, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0018_T6     | E11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 12593, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0018_T6     | E11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | E7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0018_T6     | E7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0018_T6     | E7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0018_T6     | E7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | E8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0018_T6     | E8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0018_T6     | E8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0018_T6     | E8-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E8-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E8-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E8-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | E9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | E9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | E9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | E9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | F9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | F9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | F9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | F9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | G2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0018_T6     | G2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0018_T6     | G2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0018_T6     | G3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | G5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0018_T6     | G5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0018_T6     | G5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0018_T6     | G5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | G6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0018_T6     | G6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0018_T6     | G6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0018_T6     | G7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0018_T6     | G8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | G8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0018_T6     | G8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0018_T6     | G8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 14392, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0018_T6     | G8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 14392, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | G8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 14392, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0018_T6     | G8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 14392, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0018_T6     | G8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0018_T6     | G8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0018_T6     | G8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0018_T6     | G8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0018_T6     | G8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0018_T6     | G9-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0018_T6     | G9-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0018_T6     | G9-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0018_T6     | G9-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C10-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C10-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C10-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C10-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C10-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C10-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C10-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C10-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C11-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C11-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C11-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C11-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C7-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C7-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C7-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C7-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C8-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C8-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C8-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C8-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C8-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C8-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C8-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C8-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0021_T1     | C9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0021_T1     | C9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0021_T1     | C9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0021_T1     | C9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | C9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | C9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | C9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | C9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D11-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D11-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D11-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D11-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D8-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D8-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D8-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D8-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D9-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D9-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D9-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D9-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | D9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | D9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | D9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | D9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E10-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E10-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E10-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E10-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0021_T1     | E5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0021_T1     | E5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0021_T1     | E5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0021_T1     | E5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E8-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E8-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E8-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E8-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E8-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E8-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E8-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E8-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | E9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | E9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | E9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | E9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F10-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F10-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F10-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F10-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 11051, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0021_T1     | F11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11051, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0021_T1     | F11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11051, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0021_T1     | F11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 11051, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0021_T1     | F11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F7-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F7-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F7-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F7-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F8-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F8-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F8-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F8-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F8-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F8-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F8-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F8-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F9-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F9-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F9-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F9-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | F9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | F9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | F9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | F9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G10-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G10-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G10-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G10-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G5-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G5-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G5-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G5-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0021_T1     | G5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0021_T1     | G5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0021_T1     | G5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0021_T1     | G6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G6-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G6-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G6-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G6-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0021_T1     | G8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0021_T1     | G8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0021_T1     | G8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0021_T1     | G8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G8-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G8-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G8-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G8-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G8-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G8-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G8-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G8-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0021_T1     | G9-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0021_T1     | G9-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0021_T1     | G9-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0021_T1     | G9-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0030_T1     | C11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0030_T1     | C11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0030_T1     | C11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0030_T1     | C2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | C9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | C9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | C9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | C9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0030_T1     | D2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0030_T1     | D2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0030_T1     | D2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0030_T1     | D3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0030_T1     | D4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0030_T1     | D4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0030_T1     | D4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10794, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0030_T1     | D4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0030_T1     | D5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0030_T1     | D5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0030_T1     | D5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0030_T1     | D5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | D9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | D9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | D9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | D9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0030_T1     | E4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0030_T1     | E4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0030_T1     | E4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0030_T1     | E5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 17733, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0030_T1     | E8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 17733, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0030_T1     | E8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 17733, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0030_T1     | E8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 17733, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0030_T1     | E8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | E9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | E9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | E9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | E9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | F9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | F9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | F9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | F9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0030_T1     | G4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0030_T1     | G4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0030_T1     | G4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0030_T1     | G5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0030_T1     | G5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0030_T1     | G5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0030_T1     | G5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0030_T1     | G5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8995, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0030_T1     | G9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0030_T1     | G9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0030_T1     | G9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0030_T1     | G9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C10-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C10-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C10-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C10-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7453, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | C9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | C9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | C9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | C9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | D11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | D11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | D11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | D11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | D2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | D2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | D2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | D2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | D3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | D3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | D3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | D3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | D4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | D4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | D4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | D4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | D5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | D5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | D5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | D5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | D6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | D6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | D6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | D6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | D7-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | D7-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | D7-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | D7-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | E3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | E3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | E3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | E3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | E5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | E5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | E5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | E5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | E9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | E9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | E9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | E9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | F11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | F11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | F11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | F11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | F2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | F2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | F2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | F2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | F2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | F2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | F2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | F2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | F3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | F3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | F3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | F3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | F3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | F3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | F3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | F3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | F4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | F4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | F4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | F4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | F5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | F5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | F5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | F5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | F6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 17733, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0035_T1     | F6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 17733, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0035_T1     | F6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 17733, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0035_T1     | F6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 17733, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0035_T1     | F9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | F9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | F9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | F9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0035_T1     | G7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0035_T1     | G7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0035_T1     | G7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0035_T1     | G7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B3-14      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B3-14      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B3-14      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B3-14      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B3-20      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B3-20      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B3-20      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B3-20      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B7-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B7-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B7-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B7-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | B9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | B9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | B9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | B9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | C10-13     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | C10-13     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | C10-13     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | C10-13     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | C10-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | C10-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | C10-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | C10-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | C10-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9509, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | C10-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9509, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | C10-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9509, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | C10-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9509, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | C10-8      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | C10-8      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | C10-8      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | C10-8      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | C3-12      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | C3-12      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | C3-12      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | C3-12      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | C7-12      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | C7-12      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | C7-12      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | C7-12      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | C9-15      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | C9-15      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | C9-15      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | C9-15      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D10-16     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D10-16     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D10-16     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D10-16     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D10-19     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D10-19     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D10-19     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D10-19     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D2-10      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D2-10      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D2-10      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D2-10      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D2-11      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D2-11      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D2-11      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D2-11      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D2-12      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0037_T1_CQ1 | D2-12      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0037_T1_CQ1 | D2-12      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0037_T1_CQ1 | D2-12      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0037_T1_CQ1 | D2-18      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D2-18      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D2-18      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D2-18      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D2-19      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D2-19      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D2-19      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D2-19      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D5-9       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D5-9       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D5-9       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D5-9       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | D9-14      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | D9-14      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | D9-14      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | D9-14      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | E10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | E10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | E10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | E10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | E8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | E8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | E8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | E8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | E9-10      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | E9-10      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | E9-10      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | E9-10      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F10-10     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F10-10     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F10-10     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F10-10     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F10-16     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0037_T1_CQ1 | F10-16     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0037_T1_CQ1 | F10-16     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0037_T1_CQ1 | F10-16     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0037_T1_CQ1 | F10-18     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F10-18     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F10-18     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F10-18     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F10-22     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F10-22     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F10-22     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F10-22     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F10-24     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F10-24     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F10-24     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F10-24     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F10-25     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F10-25     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F10-25     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F10-25     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F10-27     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F10-27     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F10-27     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F10-27     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F10-28     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F10-28     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F10-28     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F10-28     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F10-30     | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F10-30     | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F10-30     | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F10-30     | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F8-10      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F8-10      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F8-10      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F8-10      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F8-11      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F8-11      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F8-11      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F8-11      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F8-13      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F8-13      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F8-13      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F8-13      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F8-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F8-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F8-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F8-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F8-9       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F8-9       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F8-9       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F8-9       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F9-11      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F9-11      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F9-11      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F9-11      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F9-12      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F9-12      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F9-12      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F9-12      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | F9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | F9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | F9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | F9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | G3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | G3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | G3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | G3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | G3-11      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | G3-11      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | G3-11      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | G3-11      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | G3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | G3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | G3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | G3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0037_T1_CQ1 | G3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0037_T1_CQ1 | G3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0037_T1_CQ1 | G3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0037_T1_CQ1 | G3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B6-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B6-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B6-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B6-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 12079, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0040_T1     | B7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 12079, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0040_T1     | B7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7710, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | B9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0040_T1     | B9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0040_T1     | B9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | B9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | B9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | B9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C10-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C10-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C10-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C10-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 11051, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| NF0040_T1     | C7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11051, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0040_T1     | C7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11051, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0040_T1     | C7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 11051, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| NF0040_T1     | C7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0040_T1     | C7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0040_T1     | C7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0040_T1     | C7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 514, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0040_T1     | C7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | C8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0040_T1     | C8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0040_T1     | C9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | C9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | C9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | C9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D10-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D10-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D10-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D10-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D8-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D8-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D8-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D8-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9509, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9509, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9509, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9509, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | D9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | D9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | D9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | D9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8738, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9252, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E6-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E6-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E6-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E6-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E7-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E7-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E7-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E7-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E9-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E9-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E9-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E9-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | E9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | E9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | E9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | E9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F11-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F11-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F11-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F11-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11565, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0040_T1     | F3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11565, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| NF0040_T1     | F4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| NF0040_T1     | F7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| NF0040_T1     | F7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| NF0040_T1     | F7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| NF0040_T1     | F8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | F9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | F9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | F9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | F9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G11-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G11-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G11-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G11-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| NF0040_T1     | G7-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| NF0040_T1     | G7-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| NF0040_T1     | G7-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| NF0040_T1     | G7-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | C10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 15677, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 15677, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 15677, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 15677, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 43433, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 43433, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 43433, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 43433, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 35466, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 35466, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 35466, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 35466, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 33410, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 33410, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 33410, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 33410, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 15677, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 15677, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 15677, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 15677, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 20817, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 20817, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 20817, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 20817, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 14135, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 14135, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 14135, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 14135, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 25957, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 25957, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 25957, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 25957, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | C5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | C5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | C5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | C5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | C5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | C5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | C5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8481, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | C5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 21074, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 21074, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 21074, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 21074, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 14649, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 14649, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 14649, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 14649, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | C6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | C6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | C6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | C6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | C6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | C6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | C6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | C7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 27242, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 27242, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 27242, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 27242, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 26985, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 26985, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 26985, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 26985, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 44975, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 44975, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 44975, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 44975, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | C8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | C8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | C8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | C8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 24929, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 24929, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 24929, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 24929, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | C9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | C9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | C9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | C9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | C9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 18761, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | C9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 18761, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | C9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 18761, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | C9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 18761, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 16448, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 16448, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 16448, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 16448, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 22359, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 22359, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 22359, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 22359, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 42405, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 42405, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 42405, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 42405, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 40349, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 40349, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 40349, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 40349, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 31097, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 31097, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 31097, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 31097, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 24929, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 24929, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 24929, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 24929, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | D2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | D2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | D2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 9766, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | D3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 35466, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 35466, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 35466, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 35466, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 21331, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 21331, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 21331, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 21331, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 40863, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 40863, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 40863, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 40863, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 31097, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 31097, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 31097, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 31097, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 15163, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | D6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | D6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | D6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | D6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 26985, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 26985, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 26985, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 26985, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 35209, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 35209, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 35209, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 35209, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | D7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | D7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | D7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | D7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 283, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 283, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| SARCO219_T2   | D7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 283, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| SARCO219_T2   | D7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 283, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| SARCO219_T2   | D8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | D8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | D8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | D8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | D8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | D8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | D8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | D8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | D8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 24672, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 24672, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 24672, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 24672, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 20560, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 20560, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 20560, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 20560, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | D9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 17476, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | D9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 17476, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | D9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 17476, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | D9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 17476, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 25186, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 25186, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 25186, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 25186, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 20046, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 20046, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 20046, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 20046, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 13878, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 13878, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 13878, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 13878, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 21588, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 21588, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 21588, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 21588, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 19789, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 19789, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 19789, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 19789, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 19789, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 19789, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 19789, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 19789, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 32639, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 32639, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 32639, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 32639, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 46774, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 46774, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 46774, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 46774, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 27499, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 27499, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 27499, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 27499, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 27756, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 27756, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 27756, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 27756, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 12593, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 12593, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 12593, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 12593, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 32382, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 32382, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 32382, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 32382, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 17990, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 17990, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 17990, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 17990, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 11308, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 22873, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 22873, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 22873, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 22873, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 13621, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 32125, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | E9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 32125, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | E9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 32125, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | E9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 32125, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | E9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | E9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | E9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | E9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | F10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 39835, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 39835, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 39835, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 39835, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 21588, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 21588, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 21588, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 21588, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 18504, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 18504, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 18504, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 18504, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 30326, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 30326, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 30326, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 30326, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 17476, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 17476, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 17476, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 17476, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 32896, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 32896, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 32896, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 32896, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10280, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 15934, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 15934, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 15934, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 15934, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 18504, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 18504, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 18504, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 18504, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 46774, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 46774, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 46774, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 46774, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | F6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | F6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | F6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | F6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 20560, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 20560, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 20560, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 20560, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | F7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | F7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | F7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | F7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 38807, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 38807, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 38807, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 38807, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 23130, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 23130, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 23130, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 23130, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 18761, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 18761, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 18761, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 18761, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | F9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 31354, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | F9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 31354, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | F9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 31354, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | F9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 31354, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 39835, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 39835, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 39835, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 39835, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 33153, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 33153, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 33153, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 33153, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | G2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | G2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | G2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | G2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 29812, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 29812, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 29812, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 29812, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 32896, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 32896, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 32896, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 32896, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 24415, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 24415, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 24415, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 24415, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 29041, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 29041, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 29041, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 29041, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 18247, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 18247, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 18247, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 18247, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 37779, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 37779, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 37779, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 37779, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 30583, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 30583, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 30583, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 30583, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 19275, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 44718, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 44718, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 44718, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 44718, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10023, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 16448, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 16448, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 16448, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 16448, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 26471, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 26471, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 26471, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 26471, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 17219, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO219_T2   | G9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO219_T2   | G9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO219_T2   | G9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO219_T2   | G9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 34695, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 34695, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 34695, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 34695, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO219_T2   | G9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO219_T2   | G9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO219_T2   | G9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO219_T2   | G9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 22102, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO361_T1   | C10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C10-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C10-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C10-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C10-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C10-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C10-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C10-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C10-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO361_T1   | C11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| SARCO361_T1   | C11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| SARCO361_T1   | C11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| SARCO361_T1   | C2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO361_T1   | C3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| SARCO361_T1   | C3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| SARCO361_T1   | C3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| SARCO361_T1   | C3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C9-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C9-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C9-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C9-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | C9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | C9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | C9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | C9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D10-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D10-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D10-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D10-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D10-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D10-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D10-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D10-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D10-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D10-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D10-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D10-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D11-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D11-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D11-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D11-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D11-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D11-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D11-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D11-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 3 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO361_T1   | D2-3       | object_id_exceeds_threshold | Cell: object_id values across 3 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| SARCO361_T1   | D2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6682, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7196, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO361_T1   | D6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile           |
| SARCO361_T1   | D6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile      |
| SARCO361_T1   | D6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 771, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile   |
| SARCO361_T1   | D6-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D6-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D6-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D6-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5397, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | D9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | D9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | D9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | D9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E10-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E10-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E10-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E10-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E10-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E10-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E10-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E10-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E10-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E10-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E10-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E10-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E3-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E3-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E3-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E3-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E5-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E5-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E5-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E5-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E7-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E7-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E7-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E7-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 20817, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO361_T1   | E8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 20817, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO361_T1   | E8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 20817, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO361_T1   | E8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 20817, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO361_T1   | E8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E8-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E8-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E8-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E8-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E9-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E9-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E9-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E9-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | E9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | E9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | E9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | E9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F10-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F10-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F10-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F10-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F10-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F10-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F10-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F10-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F10-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F10-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F10-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F10-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F11-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F11-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F11-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F11-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 7967, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F2-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F2-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F2-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F2-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F3-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F3-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F3-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F3-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F3-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F3-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F3-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F3-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F3-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F3-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F3-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F3-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile       |
| SARCO361_T1   | F4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile         |
| SARCO361_T1   | F4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile    |
| SARCO361_T1   | F4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 10537, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile |
| SARCO361_T1   | F4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F4-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F4-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F4-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F4-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F5-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F5-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F5-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F5-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F5-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F5-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F5-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F5-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F5-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F5-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F5-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F5-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F5-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F5-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F5-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F5-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F6-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F6-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F6-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F6-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F6-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F6-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F6-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F6-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F8-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F8-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F8-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F8-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4883, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F8-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F8-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F8-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F8-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F9-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F9-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F9-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F9-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F9-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F9-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F9-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F9-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F9-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F9-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F9-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F9-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | F9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | F9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | F9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | F9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G10-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G10-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G10-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G10-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G10-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G10-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G10-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G10-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G10-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G10-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G10-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G10-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3598, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G10-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G10-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G10-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G10-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G10-7      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G10-7      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G10-7      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G10-7      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G11-1      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G11-1      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G11-1      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G11-1      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G11-2      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G11-2      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G11-2      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G11-2      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G11-3      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G11-3      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G11-3      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G11-3      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G11-4      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G11-4      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G11-4      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G11-4      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G11-5      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G11-5      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G11-5      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G11-5      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G11-6      | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G11-6      | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G11-6      | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G11-6      | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G2-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G2-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G2-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G2-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1542, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G2-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G2-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G2-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G2-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G2-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G2-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G2-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G2-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G2-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G2-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G2-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G2-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2827, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G2-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G2-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G2-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G2-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G2-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G2-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G2-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G2-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G3-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G3-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G3-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G3-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 8224, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G3-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G3-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G3-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G3-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G3-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G3-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G3-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G3-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3855, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G4-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G4-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G4-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G4-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G4-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G4-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G4-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G4-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G4-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G4-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G4-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G4-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G4-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G4-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G4-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G4-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G4-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G4-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G4-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G4-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G4-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G4-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G4-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G4-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G5-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G5-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G5-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G5-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G5-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G5-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G5-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G5-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G5-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G5-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G5-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G5-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G5-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G5-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G5-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G5-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6939, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G6-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G6-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G6-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G6-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4369, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G6-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G6-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G6-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G6-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4112, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G6-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G6-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G6-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G6-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1799, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G6-6       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G6-6       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G6-6       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G6-6       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G6-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G6-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G6-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G6-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6168, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G7-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G7-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G7-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G7-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1285, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G7-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G7-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G7-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G7-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 1028, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G7-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G7-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G7-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G7-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G7-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G7-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G7-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G7-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3341, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G7-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G7-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G7-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G7-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5654, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G7-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G7-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G7-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G7-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 6425, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G8-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G8-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G8-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G8-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2313, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G8-3       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G8-3       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G8-3       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G8-3       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5140, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G8-4       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G8-4       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G8-4       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G8-4       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 3084, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G8-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G8-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G8-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G8-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2056, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G8-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G8-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G8-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G8-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G9-1       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G9-1       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G9-1       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G9-1       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 2570, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G9-2       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G9-2       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G9-2       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G9-2       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G9-5       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G9-5       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G9-5       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G9-5       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 5911, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |
| SARCO361_T1   | G9-7       | object_id_exceeds_threshold | Nuclei: object_id values across 24 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile        |
| SARCO361_T1   | G9-7       | object_id_exceeds_threshold | Cell: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile          |
| SARCO361_T1   | G9-7       | object_id_exceeds_threshold | Cytoplasm: object_id values across 23 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile     |
| SARCO361_T1   | G9-7       | object_id_exceeds_threshold | Nucleocentric: object_id values across 8 input file(s) include a maximum of 4626, greater than 255 — likely the 'z_slice_global' ID scheme rather than small sequential IDs for this profile  |

## Root-causing the object-ID misalignment: image vs. feature extraction

For every well-FOV flagged with `object_id_misalignment`, the Nuclei/Cell/
Cytoplasm/Nucleocentric masks implicated in that misalignment were read directly 
and compared to **each other** — not just to their own extracted features (that's 
the source-mask consistency check below). `image_level` means the masks 
themselves already disagree, so the root cause is upstream in segmentation, not 
in this pipeline. `feature_extraction_level` means the masks agree with each 
other but the extracted feature tables don't, so the root cause is in how 
`object_id` was extracted/written per compartment. `unknown_mask_missing` means a 
mask file needed for the comparison wasn't found on disk.

| misalignment_root_cause   |   n_well_fovs |
|:--------------------------|--------------:|
| image_level               |           308 |
| feature_extraction_level  |           109 |

| patient       | well_fov   | issue_type                                | detail                                                                                                                                                                                                                                                                                                                                       |
|:--------------|:-----------|:------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| NF0014_T2     | C11-7      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2, 5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0014_T2     | C5-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | C6-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[11] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0014_T2     | C7-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | C9-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | D8-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | E8-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | F10-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | F11-6      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | F4-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | F8-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | F9-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0014_T2     | G6-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0016_T1     | C10-1      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[14] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0016_T1     | C9-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0016_T1     | E4-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0016_T1     | F3-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0018_T6     | C7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0018_T6     | D10-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0018_T6     | D5-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                  |
| NF0018_T6     | D9-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0018_T6     | E-3        | object_id_misalignment_traced_to_features | segmentation masks for ['Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                  |
| NF0018_T6     | E5-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0018_T6     | E7-3       | object_id_misalignment_traced_to_features | segmentation masks for ['Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                  |
| NF0018_T6     | F10-2      | object_id_misalignment_traced_to_features | segmentation masks for ['Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                  |
| NF0018_T6     | F8-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0018_T6     | F8-3       | object_id_misalignment_traced_to_features | segmentation masks for ['Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                  |
| NF0018_T6     | F8-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0018_T6     | G9-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[25] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0018_T6     | G9-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                  |
| NF0021_T1     | C9-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0021_T1     | E11-5      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[40] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0021_T1     | E2-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0021_T1     | F8-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[12] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0021_T1     | G11-7      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[8] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0021_T1     | G8-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0030_T1     | C10-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0030_T1     | C7-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0030_T1     | D3-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | C7-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | C7-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | D10-3      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | E11-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | E2-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | E4-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[23] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0035_T1     | F2-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | F9-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2, 5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0035_T1     | G6-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0035_T1     | G6-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | G7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | G8-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0035_T1     | G8-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0035_T1     | G9-7       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0037_T1     | B10-1      | object_id_misalignment_traced_to_features | segmentation masks for ['Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                              |
| NF0037_T1     | B10-5      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0037_T1     | B10-7      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0037_T1     | B11-7      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[3, 25] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                            |
| NF0037_T1     | B11-7      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3, 25] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                  |
| NF0037_T1     | B2-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | B2-4       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0037_T1     | B3-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0037_T1     | B3-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[20] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1     | B6-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | B6-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | B7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | B8-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | B8-6       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | B8-8       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[6, 7, 9, 10, 11, 12, 14, 15, 16, 18] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                              |
| NF0037_T1     | B8-8       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6, 7, 9, 10, 11, 12, 14, 15, 16, 18] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                    |
| NF0037_T1     | B9-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | B9-6       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | C2-3       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0037_T1     | C3-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0037_T1     | C3-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[11, 35], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                           |
| NF0037_T1     | C3-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[11, 35], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0037_T1     | C6-4       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | C7-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | C7-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[21], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1     | C7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[21], only-in-Nuclei-mask=[35] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0037_T1     | D10-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | D10-7      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | D11-2      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1     | D11-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1     | D11-3      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0037_T1     | D11-3      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | D11-6      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | D4-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | D7-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[8, 22] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                  |
| NF0037_T1     | D8-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | D8-6       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | D9-4       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | E10-2      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | E2-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[18], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1     | E2-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[18], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1     | E3-3       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | E4-5       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[6, 9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                             |
| NF0037_T1     | E4-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[6, 9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0037_T1     | E4-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[75] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1     | E5-5       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | E6-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0037_T1     | E6-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | E7-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | E9-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[9] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | E9-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[7, 19, 31] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                        |
| NF0037_T1     | E9-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[7, 19, 31] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                              |
| NF0037_T1     | F10-2      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | F11-3      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[7] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | F11-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[26] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1     | F2-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | F2-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1     | F2-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1     | F3-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | F3-3       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | F3-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1, 2, 4, 7, 12, 14, 18, 31], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                       |
| NF0037_T1     | F3-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1, 2, 4, 7, 12, 14, 18, 31], only-in-Nuclei-mask=[73] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                           |
| NF0037_T1     | F3-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | F4-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                         |
| NF0037_T1     | F5-5       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[3], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0037_T1     | F5-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[3], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1     | F6-4       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | F7-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[17, 32] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                           |
| NF0037_T1     | F7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[17, 32] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0037_T1     | F9-7       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | G10-3      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | G11-7      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | G4-4       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | G5-3       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | G5-5       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | G5-7       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | G6-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[25], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1     | G6-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[25], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1     | G7-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[11] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1     | G8-6       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[13, 22], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                           |
| NF0037_T1     | G8-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[13, 22], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0037_T1     | G9-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[16], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1     | G9-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[16], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1     | G9-5       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1     | G9-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | B10-3      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0037_T1_CQ1 | B10-3      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | B10-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[27] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B10-7      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[19], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | B10-7      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[19], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B10-8      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B2-14      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[16] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B2-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[14] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B2-8       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[28] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B2-9       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[21] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B3-10      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[32] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B3-11      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | B3-15      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[13], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | B3-15      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[13], only-in-Nuclei-mask=[28] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0037_T1_CQ1 | B3-19      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | B3-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[19] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B3-9       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[22] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B4-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4, 7] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0037_T1_CQ1 | B4-10      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[15] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B4-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[51] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B4-8       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[20], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | B4-8       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[20], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B5-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[17, 25], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                           |
| NF0037_T1_CQ1 | B5-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[17, 25], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0037_T1_CQ1 | B5-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B8-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | B8-12      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | B8-13      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[12] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B8-14      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[52], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | B8-14      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[52], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | B8-15      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | B8-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[8] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | B8-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0037_T1_CQ1 | B8-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | B8-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C10-15     | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[35], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | C10-15     | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[35], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C10-9      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | C11-1      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[20] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C2-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C3-10      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[57] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C3-11      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[30] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C3-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[24] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C3-5       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[18], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | C3-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[18], only-in-Nuclei-mask=[93] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0037_T1_CQ1 | C3-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[86] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C5-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | C6-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[13, 25] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0037_T1_CQ1 | C7-3       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | C7-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[14], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | C7-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[14], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C7-6       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | C7-8       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[7] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | C8-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | C8-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | C8-8       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[14] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C9-10      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0037_T1_CQ1 | C9-10      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | C9-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[54] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | C9-9       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0037_T1_CQ1 | C9-9       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[16] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D10-10     | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[11], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | D10-13     | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D10-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[17] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D10-20     | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D10-9      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[36] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D11-11     | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[24] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D11-3      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D11-8      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6, 13] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                  |
| NF0037_T1_CQ1 | D11-9      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[9] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | D2-17      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[14] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D2-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D2-21      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                       |
| NF0037_T1_CQ1 | D2-6       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D2-9       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[13] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D4-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D4-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[18] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D4-9       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | D5-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D5-12      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D5-17      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | D5-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | D5-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[22], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | D5-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[22], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D5-8       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[8] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | D6-10      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | D6-12      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | D6-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5, 6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0037_T1_CQ1 | D6-6       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[19, 24] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0037_T1_CQ1 | D7-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | D7-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D7-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[12] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | D8-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D8-13      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[9] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | D8-14      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[20, 27, 34], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                       |
| NF0037_T1_CQ1 | D8-14      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[20, 27, 34], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                             |
| NF0037_T1_CQ1 | D9-10      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D9-11      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D9-12      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D9-19      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | D9-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E10-11     | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | E10-6      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E10-7      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E10-9      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | E11-1      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                       |
| NF0037_T1_CQ1 | E2-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[14, 15, 24] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                             |
| NF0037_T1_CQ1 | E3-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[47] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | E4-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E4-10      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E4-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[59], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | E4-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[59], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | E4-4       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E5-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                       |
| NF0037_T1_CQ1 | E6-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E6-8       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[38] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | E6-9       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[36] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | E7-11      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E7-14      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[35] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | E7-16      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E7-25      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E7-3       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E7-7       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E8-10      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E8-8       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[20], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0037_T1_CQ1 | E8-8       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[20], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | E9-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | E9-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[8] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | E9-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[9] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | F10-11     | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F10-15     | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F10-29     | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[12] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | F10-4      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F10-7      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | F11-11     | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F11-3      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[4, 9, 23], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                         |
| NF0037_T1_CQ1 | F11-3      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[4, 9, 23], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                               |
| NF0037_T1_CQ1 | F11-9      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F2-5       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F3-17      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[28, 46], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                           |
| NF0037_T1_CQ1 | F3-17      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[28, 46], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0037_T1_CQ1 | F3-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | F4-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F4-9       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[40] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | F6-5       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F6-8       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F7-10      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5, 20] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                  |
| NF0037_T1_CQ1 | F7-12      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[20, 21] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0037_T1_CQ1 | F7-13      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F7-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | F7-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | F9-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | F9-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[17, 23] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0037_T1_CQ1 | F9-6       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | G10-1      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | G10-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | G11-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[40] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | G11-7      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[20] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | G2-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | G3-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[17] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | G4-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[17] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | G4-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[29] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | G5-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[9] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | G5-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[11] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | G6-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | G7-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | G7-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[76] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | G7-7       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | G7-9       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0037_T1_CQ1 | G8-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | G9-10      | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0037_T1_CQ1 | G9-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[18] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0037_T1_CQ1 | G9-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[15] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0040_T1     | B10-6      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | B2-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | B2-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | B2-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[22], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0040_T1     | B2-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[22], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0040_T1     | B3-4       | object_id_misalignment_traced_to_features | segmentation masks for ['Nuclei', 'Nucleocentric'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                              |
| NF0040_T1     | B3-6       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[7] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | B3-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[7] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | B6-5       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1028, 1542], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                       |
| NF0040_T1     | B6-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1028, 1542], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                             |
| NF0040_T1     | B7-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | B7-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | B7-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[771, 1028, 1285, 1542, 2056, 2313, 2570, 2827, 3084, 3341], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging           |
| NF0040_T1     | B7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[771, 1028, 1285, 1542, 2056, 2313, 2570, 2827, 3084, 3341], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging |
| NF0040_T1     | B7-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | B7-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | B9-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[257], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                              |
| NF0040_T1     | B9-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[257], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                    |
| NF0040_T1     | B9-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[257, 514, 771, 1028, 1285, 1542, 1799, 2313, 2570, 2827], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging             |
| NF0040_T1     | B9-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[257, 514, 771, 1028, 1285, 1542, 1799, 2313, 2570, 2827], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging   |
| NF0040_T1     | C10-5      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | C10-5      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | C6-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[23] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0040_T1     | C6-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1542, 2570] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                             |
| NF0040_T1     | C8-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[4, 8, 9, 13, 17, 41, 66] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                          |
| NF0040_T1     | C8-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4, 8, 9, 13, 17, 41, 66] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                |
| NF0040_T1     | C8-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[257, 1028, 1542, 2056, 2313, 2570, 3598, 3855, 4112, 4369], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging           |
| NF0040_T1     | C8-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[257, 1028, 1542, 2056, 2313, 2570, 3598, 3855, 4112, 4369], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging |
| NF0040_T1     | C8-6       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | C8-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | D2-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[13], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0040_T1     | D2-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[13], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0040_T1     | D2-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | D2-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | D2-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[1, 2, 7, 17, 24] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                  |
| NF0040_T1     | D2-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1, 2, 7, 17, 24] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                        |
| NF0040_T1     | D3-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | D3-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | D3-6       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[2, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | D3-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                            |
| NF0040_T1     | D4-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | D4-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | D5-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | D5-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | D5-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[54, 58, 63, 69] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0040_T1     | D5-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[54, 58, 63, 69] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                         |
| NF0040_T1     | D6-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[3], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | D6-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[3], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | D6-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[257, 514, 1028, 1285, 1542, 1799, 2056, 2313], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                        |
| NF0040_T1     | D6-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[257, 514, 1028, 1285, 1542, 1799, 2056, 2313], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging              |
| NF0040_T1     | D6-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[257, 514, 771, 1028, 1285, 1542, 1799, 2056], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                         |
| NF0040_T1     | D6-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[257, 514, 771, 1028, 1285, 1799, 2056], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                     |
| NF0040_T1     | D6-5       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[15], only-in-Nuclei-mask=[1, 4, 5, 6, 7, 8] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                               |
| NF0040_T1     | D6-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[15], only-in-Nuclei-mask=[1, 4, 5, 6, 7, 8, 14] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                 |
| NF0040_T1     | D6-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | D6-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | D7-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                    |
| NF0040_T1     | D7-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                          |
| NF0040_T1     | D7-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1, 3, 4, 5, 6, 7, 8, 9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                           |
| NF0040_T1     | D7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1, 3, 4, 5, 6, 7, 8, 9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                 |
| NF0040_T1     | D8-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[3, 4, 5, 7, 26] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0040_T1     | D8-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3, 4, 5, 7, 26] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                         |
| NF0040_T1     | E10-7      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[], only-in-Nuclei-mask=[3, 11, 25] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                        |
| NF0040_T1     | E10-7      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3, 11, 25] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                              |
| NF0040_T1     | E3-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[6, 9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                             |
| NF0040_T1     | E3-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[6, 9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0040_T1     | E3-5       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[6168], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                             |
| NF0040_T1     | E3-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[6168], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0040_T1     | E4-5       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[19], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0040_T1     | E4-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[19], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0040_T1     | F10-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | F10-5      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[5], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | F10-5      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[5], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | F3-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2313, 2570] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                             |
| NF0040_T1     | F3-6       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[257, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827, 3084], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging            |
| NF0040_T1     | F3-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[257, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827, 3084], only-in-Nuclei-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging  |
| NF0040_T1     | F7-6       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[514], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                              |
| NF0040_T1     | F7-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[514], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                    |
| NF0040_T1     | F8-1       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0040_T1     | G10-5      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | G2-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1542], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                             |
| NF0040_T1     | G2-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1542], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0040_T1     | G2-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2313] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                   |
| NF0040_T1     | G5-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[14], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0040_T1     | G5-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[14], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0040_T1     | G7-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | G7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0040_T1     | G9-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0040_T1     | G9-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | B11-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | B11-5      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | B11-7      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[3], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | B11-7      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[3], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | B3-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | B7-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[10, 11], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                           |
| NF0055_T1     | B7-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[10, 11], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0055_T1     | B9-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[13, 16], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                           |
| NF0055_T1     | B9-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[13, 16], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0055_T1     | C10-1      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | C10-1      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | C10-2      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[10, 18], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                           |
| NF0055_T1     | C10-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[10, 18], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| NF0055_T1     | C10-5      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[7] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | C11-3      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[12], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | C11-3      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[12], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                    |
| NF0055_T1     | C5-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[9], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | C5-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[9], only-in-Nuclei-mask=[14] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                    |
| NF0055_T1     | C5-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[13], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | C5-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[13], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | C5-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | C5-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | C7-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[13], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | C7-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[13], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | C8-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | C8-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | C8-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | C9-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | C9-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | C9-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[15], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | C9-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[15], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | D11-4      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[6], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | D11-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[6], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | D11-5      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[5], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | D11-5      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[5], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | D3-1       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | D3-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | D3-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[6], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | D3-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[6], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | D3-5       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | D3-5       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[2], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | D4-3       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | D4-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | D4-6       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[5], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | D4-6       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[5], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | D4-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | D8-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[140] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                    |
| NF0055_T1     | D8-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | D8-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | D9-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | D9-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[10], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | E11-2      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | E11-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | E11-6      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[11], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | E11-6      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[11], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | E2-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[23] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | E9-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[42], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | E9-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[42], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | F10-1      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | F10-2      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | F10-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[4], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | F11-2      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | F11-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | F4-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | F4-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | G10-4      | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[11], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | G10-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[11], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | G2-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | G2-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | G2-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[1], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | G5-2       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[19], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                               |
| NF0055_T1     | G5-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[19], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| NF0055_T1     | G5-4       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[8], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | G5-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[8], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | G5-5       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0055_T1     | G5-7       | object_id_misalignment_traced_to_image    | Cell vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cell-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                                |
| NF0055_T1     | G5-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[7], only-in-Nuclei-mask=[] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| NF0055_T1     | G7-6       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0055_T1     | G8-4       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| NF0055_T1     | G9-7       | object_id_misalignment_traced_to_features | segmentation masks for ['Cell', 'Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                          |
| SARCO219_T2   | C10-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | C3-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[13] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | C6-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | C7-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | D11-1      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[49] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | D2-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[8] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | D6-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[43] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | D6-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[36] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | D7-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[81] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | D7-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | D8-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3, 60] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                  |
| SARCO219_T2   | E10-3      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | E2-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[32] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | E2-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[58] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | E3-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[76] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | E5-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[162] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                    |
| SARCO219_T2   | E5-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[17, 66] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| SARCO219_T2   | E6-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[6] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | E6-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[58] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | E7-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[2] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | F10-4      | object_id_misalignment_traced_to_features | segmentation masks for ['Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                  |
| SARCO219_T2   | F11-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[9, 82] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                  |
| SARCO219_T2   | F2-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | F3-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[31] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | F3-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[12, 14] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| SARCO219_T2   | F5-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[60] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | F5-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[8] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | F8-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | F9-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO219_T2   | F9-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[45] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | G11-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[44] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | G11-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[29, 30] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                 |
| SARCO219_T2   | G2-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1, 10] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                  |
| SARCO219_T2   | G4-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[56] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | G5-2       | object_id_misalignment_traced_to_features | segmentation masks for ['Cytoplasm', 'Nuclei'] agree with each other on object IDs, but the extracted feature tables for these compartments disagree — the misalignment was introduced during feature extraction or merging, not in the source masks/images                                                                                  |
| SARCO219_T2   | G8-2       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[19] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO219_T2   | G9-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[5, 52] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                  |
| SARCO361_T1   | C2-7       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[4] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO361_T1   | C5-4       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[23] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO361_T1   | C9-3       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[8] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO361_T1   | D10-4      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[3] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO361_T1   | D2-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[12] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                     |
| SARCO361_T1   | D6-1       | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |
| SARCO361_T1   | E10-2      | object_id_misalignment_traced_to_image    | Cytoplasm vs Nuclei: the segmentation masks themselves disagree on object IDs — only-in-Cytoplasm-mask=[], only-in-Nuclei-mask=[1] — this well-FOV's object_id misalignment originates in the source masks/images, not in feature extraction or merging                                                                                      |

## Source mask consistency check

For every well-FOV/compartment implicated in a merge-related issue (blow-up, 
duplicate object_id, a failed Organoid merge, an object ID over 
255, or cross-compartment misalignment), the 
segmentation mask TIFF was read directly and its unique object IDs compared 
against the object IDs in the extracted features. A mismatch points to the mask 
on disk (likely re-segmented/overwritten after featurization last ran) rather 
than a merge bug.

| patient       | well_fov   | issue_type                     | detail                                                                                                                                                                                                                                                                                                                                                                                                             |
|:--------------|:-----------|:-------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| NF0016_T1     | C10-1      | source_mask_object_id_mismatch | Cell: mask has 19 object IDs, extracted features have 19 — only-in-mask=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| NF0016_T1     | C10-1      | source_mask_object_id_mismatch | Cytoplasm: mask has 18 object IDs, extracted features have 18 — only-in-mask=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| NF0016_T1     | C10-1      | source_mask_object_id_mismatch | Nuclei: mask has 19 object IDs, extracted features have 19 — only-in-mask=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug             |
| NF0016_T1     | C10-1      | source_mask_object_id_mismatch | Nucleocentric: mask has 19 object IDs, extracted features have 19 — only-in-mask=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug      |
| NF0016_T1     | E4-1       | source_mask_object_id_mismatch | Cell: mask has 13 object IDs, extracted features have 13 — only-in-mask=[1, 4, 5, 6, 7, 9, 10, 12, 13, 14], only-in-features=[257, 1028, 1285, 1542, 1799, 2313, 2570, 3084, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| NF0016_T1     | E4-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 12 object IDs, extracted features have 12 — only-in-mask=[1, 5, 6, 7, 9, 10, 12, 13, 14, 15], only-in-features=[257, 1285, 1542, 1799, 2313, 2570, 3084, 3341, 3598, 3855] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug      |
| NF0016_T1     | E4-1       | source_mask_object_id_mismatch | Nuclei: mask has 13 object IDs, extracted features have 13 — only-in-mask=[1, 4, 5, 6, 7, 9, 10, 12, 13, 14], only-in-features=[257, 1028, 1285, 1542, 1799, 2313, 2570, 3084, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| NF0016_T1     | E4-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 13 object IDs, extracted features have 13 — only-in-mask=[1, 4, 5, 6, 7, 9, 10, 12, 13, 14], only-in-features=[257, 1028, 1285, 1542, 1799, 2313, 2570, 3084, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug   |
| NF0018_T6     | D5-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 10 object IDs, extracted features have 19 — only-in-mask=[], only-in-features=[514, 1028, 2056, 2313, 2570, 2827, 3084, 3855, 4369] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                             |
| NF0018_T6     | E-3        | source_mask_object_id_mismatch | Cytoplasm: mask has 13 object IDs, extracted features have 26 — only-in-mask=[], only-in-features=[514, 771, 1285, 2056, 2313, 3084, 4112, 4883, 5397, 5654] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                        |
| NF0018_T6     | E7-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 9 object IDs, extracted features have 18 — only-in-mask=[], only-in-features=[514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                               |
| NF0018_T6     | F10-2      | source_mask_object_id_mismatch | Cytoplasm: mask has 23 object IDs, extracted features have 46 — only-in-mask=[], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                         |
| NF0018_T6     | F8-1       | source_mask_object_id_mismatch | Cell: mask has 5 object IDs, extracted features have 5 — only-in-mask=[1, 2, 3, 5, 6], only-in-features=[257, 514, 771, 1285, 1542] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                 |
| NF0018_T6     | F8-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 4 object IDs, extracted features have 4 — only-in-mask=[2, 3, 5, 6], only-in-features=[514, 771, 1285, 1542] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                    |
| NF0018_T6     | F8-1       | source_mask_object_id_mismatch | Nuclei: mask has 5 object IDs, extracted features have 5 — only-in-mask=[1, 2, 3, 5, 6], only-in-features=[257, 514, 771, 1285, 1542] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                               |
| NF0018_T6     | F8-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 5 object IDs, extracted features have 5 — only-in-mask=[1, 2, 3, 5, 6], only-in-features=[257, 514, 771, 1285, 1542] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                        |
| NF0018_T6     | F8-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 4 object IDs, extracted features have 8 — only-in-mask=[], only-in-features=[257, 514, 771, 1028] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                               |
| NF0018_T6     | F8-4       | source_mask_object_id_mismatch | Cell: mask has 5 object IDs, extracted features have 5 — only-in-mask=[1, 2, 3, 4, 6], only-in-features=[257, 514, 771, 1028, 1542] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                 |
| NF0018_T6     | F8-4       | source_mask_object_id_mismatch | Cytoplasm: mask has 4 object IDs, extracted features have 4 — only-in-mask=[1, 3, 4, 6], only-in-features=[257, 771, 1028, 1542] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                    |
| NF0018_T6     | F8-4       | source_mask_object_id_mismatch | Nuclei: mask has 5 object IDs, extracted features have 5 — only-in-mask=[1, 2, 3, 4, 6], only-in-features=[257, 514, 771, 1028, 1542] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                               |
| NF0018_T6     | F8-4       | source_mask_object_id_mismatch | Nucleocentric: mask has 5 object IDs, extracted features have 5 — only-in-mask=[1, 2, 3, 4, 6], only-in-features=[257, 514, 771, 1028, 1542] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                        |
| NF0018_T6     | G9-1       | source_mask_object_id_mismatch | Cell: mask has 25 object IDs, extracted features have 25 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                 |
| NF0018_T6     | G9-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 24 object IDs, extracted features have 24 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| NF0018_T6     | G9-1       | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 25 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| NF0018_T6     | G9-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 25 object IDs, extracted features have 25 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| NF0018_T6     | G9-2       | source_mask_object_id_mismatch | Cytoplasm: mask has 34 object IDs, extracted features have 68 — only-in-mask=[], only-in-features=[257, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                        |
| NF0021_T1     | E2-1       | source_mask_object_id_mismatch | Cell: mask has 21 object IDs, extracted features have 21 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                |
| NF0021_T1     | E2-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 20 object IDs, extracted features have 20 — only-in-mask=[1, 3, 4, 5, 7, 8, 9, 10, 11, 12], only-in-features=[257, 771, 1028, 1285, 1799, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| NF0021_T1     | E2-1       | source_mask_object_id_mismatch | Nuclei: mask has 21 object IDs, extracted features have 21 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug              |
| NF0021_T1     | E2-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 21 object IDs, extracted features have 21 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug       |
| NF0021_T1     | F8-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 41 object IDs, extracted features have 43 — only-in-mask=[13, 15, 19, 28], only-in-features=[12, 14, 17, 27, 47, 48] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                            |
| NF0021_T1     | F8-1       | source_mask_object_id_mismatch | Nuclei: mask has 42 object IDs, extracted features have 44 — only-in-mask=[15, 19, 28], only-in-features=[14, 17, 27, 47, 48] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                       |
| NF0021_T1     | G11-7      | source_mask_object_id_mismatch | Cell: mask has 8 object IDs, extracted features have 8 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                      |
| NF0021_T1     | G11-7      | source_mask_object_id_mismatch | Cytoplasm: mask has 7 object IDs, extracted features have 7 — only-in-mask=[1, 2, 3, 4, 5, 6, 7], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                          |
| NF0021_T1     | G11-7      | source_mask_object_id_mismatch | Nuclei: mask has 8 object IDs, extracted features have 8 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                    |
| NF0021_T1     | G11-7      | source_mask_object_id_mismatch | Nucleocentric: mask has 8 object IDs, extracted features have 8 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                             |
| NF0030_T1     | C7-2       | source_mask_object_id_mismatch | Cell: mask has 21 object IDs, extracted features have 21 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                 |
| NF0030_T1     | C7-2       | source_mask_object_id_mismatch | Cytoplasm: mask has 20 object IDs, extracted features have 20 — only-in-mask=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| NF0030_T1     | C7-2       | source_mask_object_id_mismatch | Nuclei: mask has 21 object IDs, extracted features have 21 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| NF0030_T1     | C7-2       | source_mask_object_id_mismatch | Nucleocentric: mask has 21 object IDs, extracted features have 21 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| NF0030_T1     | D3-1       | source_mask_object_id_mismatch | Cell: mask has 13 object IDs, extracted features have 13 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                 |
| NF0030_T1     | D3-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 12 object IDs, extracted features have 12 — only-in-mask=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| NF0030_T1     | D3-1       | source_mask_object_id_mismatch | Nuclei: mask has 13 object IDs, extracted features have 13 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| NF0030_T1     | D3-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 13 object IDs, extracted features have 13 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| NF0035_T1     | G6-2       | source_mask_object_id_mismatch | Nuclei: mask has 18 object IDs, extracted features have 15 — only-in-mask=[3, 4, 11], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                           |
| NF0035_T1     | G6-2       | source_mask_object_id_mismatch | Nucleocentric: mask has 18 object IDs, extracted features have 21 — only-in-mask=[], only-in-features=[19, 20, 21] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                  |
| NF0035_T1     | G6-7       | source_mask_object_id_mismatch | Cytoplasm: mask has 8 object IDs, extracted features have 9 — only-in-mask=[], only-in-features=[2] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0035_T1     | G6-7       | source_mask_object_id_mismatch | Nuclei: mask has 9 object IDs, extracted features have 8 — only-in-mask=[2], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                    |
| NF0035_T1     | G6-7       | source_mask_object_id_mismatch | Nucleocentric: mask has 9 object IDs, extracted features have 10 — only-in-mask=[2], only-in-features=[10, 11] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                      |
| NF0035_T1     | G7-3       | source_mask_object_id_mismatch | Cell: mask has 10 object IDs, extracted features have 9 — only-in-mask=[1, 2, 4, 5, 7, 8, 9, 10, 11, 12], only-in-features=[257, 514, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                     |
| NF0035_T1     | G7-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 9 object IDs, extracted features have 8 — only-in-mask=[1, 4, 5, 7, 8, 9, 10, 11, 12], only-in-features=[257, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                         |
| NF0035_T1     | G7-3       | source_mask_object_id_mismatch | Nuclei: mask has 10 object IDs, extracted features have 9 — only-in-mask=[1, 2, 4, 5, 7, 8, 9, 10, 11, 12], only-in-features=[257, 514, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                   |
| NF0035_T1     | G7-3       | source_mask_object_id_mismatch | Nucleocentric: mask has 10 object IDs, extracted features have 9 — only-in-mask=[1, 2, 4, 5, 7, 8, 9, 10, 11, 12], only-in-features=[257, 514, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| NF0035_T1     | G8-2       | source_mask_object_id_mismatch | Nuclei: mask has 16 object IDs, extracted features have 15 — only-in-mask=[9], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0035_T1     | G8-2       | source_mask_object_id_mismatch | Nucleocentric: mask has 16 object IDs, extracted features have 17 — only-in-mask=[6, 14], only-in-features=[15, 18, 19] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                             |
| NF0035_T1     | G9-7       | source_mask_object_id_mismatch | Nuclei: mask has 10 object IDs, extracted features have 7 — only-in-mask=[1, 7, 8], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0035_T1     | G9-7       | source_mask_object_id_mismatch | Nucleocentric: mask has 10 object IDs, extracted features have 8 — only-in-mask=[8, 11], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                        |
| NF0037_T1     | B10-1      | source_mask_object_id_mismatch | Nucleocentric: mask has 45 object IDs, extracted features have 52 — only-in-mask=[], only-in-features=[8, 47, 48, 49, 50, 51, 52] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                   |
| NF0037_T1     | B10-5      | source_mask_object_id_mismatch | Nuclei: mask has 22 object IDs, extracted features have 21 — only-in-mask=[5], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | B10-5      | source_mask_object_id_mismatch | Nucleocentric: mask has 22 object IDs, extracted features have 20 — only-in-mask=[21, 22], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                      |
| NF0037_T1     | B10-7      | source_mask_object_id_mismatch | Cell: mask has 25 object IDs, extracted features have 23 — only-in-mask=[25, 26, 27], only-in-features=[7] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                          |
| NF0037_T1     | B10-7      | source_mask_object_id_mismatch | Cytoplasm: mask has 25 object IDs, extracted features have 23 — only-in-mask=[25, 26, 27], only-in-features=[7] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                     |
| NF0037_T1     | B10-7      | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 17 — only-in-mask=[16, 18, 19, 20, 21, 22, 25, 26, 27], only-in-features=[7] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                |
| NF0037_T1     | B10-7      | source_mask_object_id_mismatch | Nucleocentric: mask has 25 object IDs, extracted features have 22 — only-in-mask=[4, 13, 18, 26, 27], only-in-features=[3, 7] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                       |
| NF0037_T1     | B2-4       | source_mask_object_id_mismatch | Nuclei: mask has 52 object IDs, extracted features have 61 — only-in-mask=[], only-in-features=[2, 3, 14, 16, 17, 26, 50, 51, 61] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                   |
| NF0037_T1     | B3-2       | source_mask_object_id_mismatch | Nuclei: mask has 41 object IDs, extracted features have 45 — only-in-mask=[], only-in-features=[4, 9, 13, 17] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                       |
| NF0037_T1     | B6-2       | source_mask_object_id_mismatch | Nuclei: mask has 24 object IDs, extracted features have 23 — only-in-mask=[7], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | B8-1       | source_mask_object_id_mismatch | Nuclei: mask has 22 object IDs, extracted features have 21 — only-in-mask=[8], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | B8-6       | source_mask_object_id_mismatch | Nuclei: mask has 18 object IDs, extracted features have 17 — only-in-mask=[9], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | B9-2       | source_mask_object_id_mismatch | Nuclei: mask has 9 object IDs, extracted features have 8 — only-in-mask=[7], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                    |
| NF0037_T1     | B9-6       | source_mask_object_id_mismatch | Nuclei: mask has 18 object IDs, extracted features have 17 — only-in-mask=[13], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1     | C2-3       | source_mask_object_id_mismatch | Nuclei: mask has 70 object IDs, extracted features have 71 — only-in-mask=[], only-in-features=[38] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1     | C3-1       | source_mask_object_id_mismatch | Nuclei: mask has 53 object IDs, extracted features have 55 — only-in-mask=[], only-in-features=[7, 55] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1     | C6-4       | source_mask_object_id_mismatch | Cell: mask has 19 object IDs, extracted features have 19 — only-in-mask=[11], only-in-features=[5] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | C6-4       | source_mask_object_id_mismatch | Cytoplasm: mask has 19 object IDs, extracted features have 19 — only-in-mask=[11], only-in-features=[5] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0037_T1     | C6-4       | source_mask_object_id_mismatch | Nuclei: mask has 19 object IDs, extracted features have 18 — only-in-mask=[7, 11], only-in-features=[5] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0037_T1     | C7-2       | source_mask_object_id_mismatch | Nuclei: mask has 15 object IDs, extracted features have 9 — only-in-mask=[1, 2, 8, 9, 12, 13], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                  |
| NF0037_T1     | D10-7      | source_mask_object_id_mismatch | Nuclei: mask has 20 object IDs, extracted features have 19 — only-in-mask=[16], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1     | D11-6      | source_mask_object_id_mismatch | Nuclei: mask has 15 object IDs, extracted features have 11 — only-in-mask=[1, 2, 3, 6], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                         |
| NF0037_T1     | D8-6       | source_mask_object_id_mismatch | Cell: mask has 16 object IDs, extracted features have 14 — only-in-mask=[1, 12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                |
| NF0037_T1     | D8-6       | source_mask_object_id_mismatch | Cytoplasm: mask has 16 object IDs, extracted features have 14 — only-in-mask=[1, 12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                           |
| NF0037_T1     | D8-6       | source_mask_object_id_mismatch | Nuclei: mask has 16 object IDs, extracted features have 13 — only-in-mask=[1, 5, 12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                           |
| NF0037_T1     | D9-4       | source_mask_object_id_mismatch | Nuclei: mask has 24 object IDs, extracted features have 23 — only-in-mask=[1], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | E10-2      | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 24 — only-in-mask=[10], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1     | E3-3       | source_mask_object_id_mismatch | Nuclei: mask has 28 object IDs, extracted features have 21 — only-in-mask=[9, 10, 12, 13, 21, 24, 27], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                          |
| NF0037_T1     | E4-7       | source_mask_object_id_mismatch | Nuclei: mask has 75 object IDs, extracted features have 76 — only-in-mask=[], only-in-features=[62] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1     | E5-5       | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 24 — only-in-mask=[5], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | E7-2       | source_mask_object_id_mismatch | Nuclei: mask has 16 object IDs, extracted features have 15 — only-in-mask=[3], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | F10-2      | source_mask_object_id_mismatch | Nuclei: mask has 11 object IDs, extracted features have 10 — only-in-mask=[1], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | F11-3      | source_mask_object_id_mismatch | Cytoplasm: mask has 24 object IDs, extracted features have 22 — only-in-mask=[3, 4], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                            |
| NF0037_T1     | F11-3      | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 23 — only-in-mask=[3, 4], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                               |
| NF0037_T1     | F11-4      | source_mask_object_id_mismatch | Cytoplasm: mask has 24 object IDs, extracted features have 24 — only-in-mask=[13, 15], only-in-features=[14, 16] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                    |
| NF0037_T1     | F11-4      | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 25 — only-in-mask=[13, 15], only-in-features=[14, 16] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                       |
| NF0037_T1     | F3-1       | source_mask_object_id_mismatch | Nuclei: mask has 21 object IDs, extracted features have 20 — only-in-mask=[16], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1     | F3-3       | source_mask_object_id_mismatch | Nuclei: mask has 28 object IDs, extracted features have 27 — only-in-mask=[12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1     | F4-2       | source_mask_object_id_mismatch | Nuclei: mask has 46 object IDs, extracted features have 50 — only-in-mask=[], only-in-features=[19, 30, 37, 38] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                     |
| NF0037_T1     | F6-4       | source_mask_object_id_mismatch | Nuclei: mask has 7 object IDs, extracted features have 6 — only-in-mask=[2], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                    |
| NF0037_T1     | F9-7       | source_mask_object_id_mismatch | Nuclei: mask has 19 object IDs, extracted features have 18 — only-in-mask=[8], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | G10-3      | source_mask_object_id_mismatch | Nuclei: mask has 16 object IDs, extracted features have 14 — only-in-mask=[6, 9], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                               |
| NF0037_T1     | G11-7      | source_mask_object_id_mismatch | Nuclei: mask has 22 object IDs, extracted features have 20 — only-in-mask=[12, 20], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0037_T1     | G4-4       | source_mask_object_id_mismatch | Nuclei: mask has 16 object IDs, extracted features have 15 — only-in-mask=[1], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | G5-3       | source_mask_object_id_mismatch | Cell: mask has 24 object IDs, extracted features have 24 — only-in-mask=[14], only-in-features=[13] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1     | G5-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 24 object IDs, extracted features have 24 — only-in-mask=[14], only-in-features=[13] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                            |
| NF0037_T1     | G5-3       | source_mask_object_id_mismatch | Nuclei: mask has 24 object IDs, extracted features have 23 — only-in-mask=[3, 14], only-in-features=[13] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                            |
| NF0037_T1     | G5-5       | source_mask_object_id_mismatch | Nuclei: mask has 27 object IDs, extracted features have 26 — only-in-mask=[9], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1     | G5-7       | source_mask_object_id_mismatch | Nuclei: mask has 21 object IDs, extracted features have 19 — only-in-mask=[2, 13], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1     | G9-5       | source_mask_object_id_mismatch | Nuclei: mask has 14 object IDs, extracted features have 13 — only-in-mask=[1], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | B3-10      | source_mask_object_id_mismatch | Cell: mask has 24 object IDs, extracted features have 23 — only-in-mask=[19], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                   |
| NF0037_T1_CQ1 | B3-10      | source_mask_object_id_mismatch | Cytoplasm: mask has 23 object IDs, extracted features have 22 — only-in-mask=[19], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1_CQ1 | B3-10      | source_mask_object_id_mismatch | Nuclei: mask has 24 object IDs, extracted features have 22 — only-in-mask=[1, 19], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1_CQ1 | B3-11      | source_mask_object_id_mismatch | Nuclei: mask has 10 object IDs, extracted features have 9 — only-in-mask=[5], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                   |
| NF0037_T1_CQ1 | B3-19      | source_mask_object_id_mismatch | Nuclei: mask has 28 object IDs, extracted features have 27 — only-in-mask=[11], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | B4-10      | source_mask_object_id_mismatch | Nuclei: mask has 28 object IDs, extracted features have 26 — only-in-mask=[13, 15], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0037_T1_CQ1 | B8-1       | source_mask_object_id_mismatch | Nuclei: mask has 43 object IDs, extracted features have 42 — only-in-mask=[34], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | B8-12      | source_mask_object_id_mismatch | Nuclei: mask has 24 object IDs, extracted features have 23 — only-in-mask=[6], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | B8-15      | source_mask_object_id_mismatch | Nuclei: mask has 24 object IDs, extracted features have 22 — only-in-mask=[8, 14], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1_CQ1 | B8-2       | source_mask_object_id_mismatch | Nuclei: mask has 34 object IDs, extracted features have 33 — only-in-mask=[2], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | C10-9      | source_mask_object_id_mismatch | Nuclei: mask has 29 object IDs, extracted features have 28 — only-in-mask=[27], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | C3-11      | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 24 — only-in-mask=[11], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | C3-5       | source_mask_object_id_mismatch | Cell: mask has 187 object IDs, extracted features have 187 — only-in-mask=[82, 96], only-in-features=[81, 83] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                       |
| NF0037_T1_CQ1 | C3-5       | source_mask_object_id_mismatch | Cytoplasm: mask has 186 object IDs, extracted features have 186 — only-in-mask=[82, 92, 96], only-in-features=[81, 83, 93] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                          |
| NF0037_T1_CQ1 | C3-5       | source_mask_object_id_mismatch | Nuclei: mask has 186 object IDs, extracted features have 188 — only-in-mask=[], only-in-features=[81, 83] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                           |
| NF0037_T1_CQ1 | C3-5       | source_mask_object_id_mismatch | Nucleocentric: mask has 186 object IDs, extracted features have 186 — only-in-mask=[82, 96], only-in-features=[81, 83] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                              |
| NF0037_T1_CQ1 | C7-3       | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 24 — only-in-mask=[22], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | C7-6       | source_mask_object_id_mismatch | Nuclei: mask has 14 object IDs, extracted features have 13 — only-in-mask=[4], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | C8-2       | source_mask_object_id_mismatch | Nuclei: mask has 17 object IDs, extracted features have 16 — only-in-mask=[14], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D10-13     | source_mask_object_id_mismatch | Nuclei: mask has 26 object IDs, extracted features have 24 — only-in-mask=[27, 31], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0037_T1_CQ1 | D10-2      | source_mask_object_id_mismatch | Nuclei: mask has 16 object IDs, extracted features have 15 — only-in-mask=[15], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D10-20     | source_mask_object_id_mismatch | Nuclei: mask has 13 object IDs, extracted features have 12 — only-in-mask=[11], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D11-3      | source_mask_object_id_mismatch | Nuclei: mask has 27 object IDs, extracted features have 25 — only-in-mask=[9, 20], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1_CQ1 | D2-2       | source_mask_object_id_mismatch | Nuclei: mask has 18 object IDs, extracted features have 17 — only-in-mask=[12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D2-21      | source_mask_object_id_mismatch | Nuclei: mask has 27 object IDs, extracted features have 26 — only-in-mask=[19], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D2-6       | source_mask_object_id_mismatch | Nuclei: mask has 12 object IDs, extracted features have 11 — only-in-mask=[11], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D4-2       | source_mask_object_id_mismatch | Nuclei: mask has 18 object IDs, extracted features have 14 — only-in-mask=[6, 8, 9, 11], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                        |
| NF0037_T1_CQ1 | D5-1       | source_mask_object_id_mismatch | Nuclei: mask has 28 object IDs, extracted features have 27 — only-in-mask=[13], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D5-12      | source_mask_object_id_mismatch | Nuclei: mask has 20 object IDs, extracted features have 19 — only-in-mask=[9], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | D6-6       | source_mask_object_id_mismatch | Nuclei: mask has 8 object IDs, extracted features have 7 — only-in-mask=[1], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                    |
| NF0037_T1_CQ1 | D8-1       | source_mask_object_id_mismatch | Nuclei: mask has 9 object IDs, extracted features have 7 — only-in-mask=[8, 9], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D9-10      | source_mask_object_id_mismatch | Nuclei: mask has 14 object IDs, extracted features have 13 — only-in-mask=[15], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D9-11      | source_mask_object_id_mismatch | Nuclei: mask has 22 object IDs, extracted features have 21 — only-in-mask=[8], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | D9-12      | source_mask_object_id_mismatch | Nuclei: mask has 22 object IDs, extracted features have 21 — only-in-mask=[1], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | D9-19      | source_mask_object_id_mismatch | Nuclei: mask has 19 object IDs, extracted features have 18 — only-in-mask=[17], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | D9-2       | source_mask_object_id_mismatch | Nuclei: mask has 14 object IDs, extracted features have 13 — only-in-mask=[14], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | E10-6      | source_mask_object_id_mismatch | Nuclei: mask has 21 object IDs, extracted features have 20 — only-in-mask=[13], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | E10-7      | source_mask_object_id_mismatch | Nuclei: mask has 14 object IDs, extracted features have 13 — only-in-mask=[14], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | E11-1      | source_mask_object_id_mismatch | Nuclei: mask has 30 object IDs, extracted features have 29 — only-in-mask=[13], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | E2-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 19 object IDs, extracted features have 17 — only-in-mask=[8, 16], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                           |
| NF0037_T1_CQ1 | E2-3       | source_mask_object_id_mismatch | Nuclei: mask has 22 object IDs, extracted features have 20 — only-in-mask=[8, 16], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1_CQ1 | E4-1       | source_mask_object_id_mismatch | Nuclei: mask has 26 object IDs, extracted features have 25 — only-in-mask=[19], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | E4-10      | source_mask_object_id_mismatch | Nuclei: mask has 30 object IDs, extracted features have 29 — only-in-mask=[7], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | E4-4       | source_mask_object_id_mismatch | Nuclei: mask has 30 object IDs, extracted features have 29 — only-in-mask=[18], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | E5-1       | source_mask_object_id_mismatch | Nuclei: mask has 17 object IDs, extracted features have 16 — only-in-mask=[6], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | E6-2       | source_mask_object_id_mismatch | Nuclei: mask has 29 object IDs, extracted features have 27 — only-in-mask=[16, 21], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0037_T1_CQ1 | E7-11      | source_mask_object_id_mismatch | Nuclei: mask has 11 object IDs, extracted features have 10 — only-in-mask=[16], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | E7-16      | source_mask_object_id_mismatch | Cell: mask has 24 object IDs, extracted features have 23 — only-in-mask=[12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                   |
| NF0037_T1_CQ1 | E7-16      | source_mask_object_id_mismatch | Cytoplasm: mask has 24 object IDs, extracted features have 23 — only-in-mask=[12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1_CQ1 | E7-16      | source_mask_object_id_mismatch | Nuclei: mask has 24 object IDs, extracted features have 22 — only-in-mask=[12, 14], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0037_T1_CQ1 | E7-25      | source_mask_object_id_mismatch | Nuclei: mask has 17 object IDs, extracted features have 15 — only-in-mask=[4, 13], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1_CQ1 | E7-3       | source_mask_object_id_mismatch | Nuclei: mask has 7 object IDs, extracted features have 6 — only-in-mask=[2], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                    |
| NF0037_T1_CQ1 | E7-7       | source_mask_object_id_mismatch | Nuclei: mask has 4 object IDs, extracted features have 3 — only-in-mask=[19], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                   |
| NF0037_T1_CQ1 | E8-10      | source_mask_object_id_mismatch | Nuclei: mask has 22 object IDs, extracted features have 21 — only-in-mask=[5], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | E9-1       | source_mask_object_id_mismatch | Nuclei: mask has 19 object IDs, extracted features have 18 — only-in-mask=[7], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | E9-6       | source_mask_object_id_mismatch | Nuclei: mask has 39 object IDs, extracted features have 38 — only-in-mask=[12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | F10-11     | source_mask_object_id_mismatch | Nuclei: mask has 3 object IDs, extracted features have 2 — only-in-mask=[12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                   |
| NF0037_T1_CQ1 | F10-15     | source_mask_object_id_mismatch | Nuclei: mask has 6 object IDs, extracted features have 5 — only-in-mask=[6], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                    |
| NF0037_T1_CQ1 | F10-4      | source_mask_object_id_mismatch | Nuclei: mask has 14 object IDs, extracted features have 13 — only-in-mask=[4], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | F11-11     | source_mask_object_id_mismatch | Nuclei: mask has 17 object IDs, extracted features have 13 — only-in-mask=[11, 12, 13, 15], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                     |
| NF0037_T1_CQ1 | F11-9      | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 24 — only-in-mask=[12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | F2-5       | source_mask_object_id_mismatch | Nuclei: mask has 10 object IDs, extracted features have 9 — only-in-mask=[8], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                   |
| NF0037_T1_CQ1 | F4-1       | source_mask_object_id_mismatch | Nuclei: mask has 28 object IDs, extracted features have 26 — only-in-mask=[3, 8], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                               |
| NF0037_T1_CQ1 | F6-5       | source_mask_object_id_mismatch | Nuclei: mask has 17 object IDs, extracted features have 16 — only-in-mask=[5], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | F6-8       | source_mask_object_id_mismatch | Nuclei: mask has 27 object IDs, extracted features have 26 — only-in-mask=[7], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | F7-10      | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 21 — only-in-mask=[20, 22, 23, 24], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                     |
| NF0037_T1_CQ1 | F7-12      | source_mask_object_id_mismatch | Nuclei: mask has 27 object IDs, extracted features have 25 — only-in-mask=[1, 14], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| NF0037_T1_CQ1 | F7-13      | source_mask_object_id_mismatch | Nuclei: mask has 14 object IDs, extracted features have 13 — only-in-mask=[9], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | F9-2       | source_mask_object_id_mismatch | Nuclei: mask has 29 object IDs, extracted features have 28 — only-in-mask=[1], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | F9-6       | source_mask_object_id_mismatch | Nuclei: mask has 20 object IDs, extracted features have 19 — only-in-mask=[5], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | G10-1      | source_mask_object_id_mismatch | Nuclei: mask has 24 object IDs, extracted features have 23 — only-in-mask=[5], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | G2-2       | source_mask_object_id_mismatch | Nuclei: mask has 19 object IDs, extracted features have 18 — only-in-mask=[6], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | G3-5       | source_mask_object_id_mismatch | Cytoplasm: mask has 16 object IDs, extracted features have 14 — only-in-mask=[19, 26], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                          |
| NF0037_T1_CQ1 | G3-5       | source_mask_object_id_mismatch | Nuclei: mask has 17 object IDs, extracted features have 15 — only-in-mask=[19, 26], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0037_T1_CQ1 | G7-1       | source_mask_object_id_mismatch | Nuclei: mask has 29 object IDs, extracted features have 28 — only-in-mask=[12], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0037_T1_CQ1 | G7-7       | source_mask_object_id_mismatch | Nuclei: mask has 20 object IDs, extracted features have 19 — only-in-mask=[5], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | G8-2       | source_mask_object_id_mismatch | Nuclei: mask has 13 object IDs, extracted features have 12 — only-in-mask=[2], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0037_T1_CQ1 | G9-10      | source_mask_object_id_mismatch | Nuclei: mask has 14 object IDs, extracted features have 13 — only-in-mask=[6], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0040_T1     | B3-4       | source_mask_object_id_mismatch | Nuclei: mask has 22 object IDs, extracted features have 23 — only-in-mask=[], only-in-features=[1] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0040_T1     | F8-1       | source_mask_object_id_mismatch | Nuclei: mask has 35 object IDs, extracted features have 37 — only-in-mask=[], only-in-features=[32, 33] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                             |
| NF0055_T1     | G5-5       | source_mask_object_id_mismatch | Nuclei: mask has 10 object IDs, extracted features have 8 — only-in-mask=[5, 6], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                |
| NF0055_T1     | G7-6       | source_mask_object_id_mismatch | Nuclei: mask has 12 object IDs, extracted features have 11 — only-in-mask=[4], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                  |
| NF0055_T1     | G8-4       | source_mask_object_id_mismatch | Nuclei: mask has 36 object IDs, extracted features have 35 — only-in-mask=[10], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| NF0055_T1     | G9-7       | source_mask_object_id_mismatch | Nuclei: mask has 8 object IDs, extracted features have 7 — only-in-mask=[2], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                    |
| SARCO219_T2   | C3-1       | source_mask_object_id_mismatch | Cell: mask has 54 object IDs, extracted features have 52 — only-in-mask=[1, 2, 3, 4, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                |
| SARCO219_T2   | C3-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 53 object IDs, extracted features have 51 — only-in-mask=[1, 2, 3, 4, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug           |
| SARCO219_T2   | C3-1       | source_mask_object_id_mismatch | Nuclei: mask has 54 object IDs, extracted features have 52 — only-in-mask=[1, 2, 3, 4, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug              |
| SARCO219_T2   | C3-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 54 object IDs, extracted features have 52 — only-in-mask=[1, 2, 3, 4, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug       |
| SARCO219_T2   | C6-3       | source_mask_object_id_mismatch | Cell: mask has 26 object IDs, extracted features have 25 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 10, 11, 13], only-in-features=[257, 514, 1028, 1285, 1542, 1799, 2056, 2570, 2827, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug              |
| SARCO219_T2   | C6-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 25 object IDs, extracted features have 24 — only-in-mask=[1, 2, 4, 5, 7, 8, 10, 11, 13, 14], only-in-features=[257, 514, 1028, 1285, 1799, 2056, 2570, 2827, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| SARCO219_T2   | C6-3       | source_mask_object_id_mismatch | Nuclei: mask has 26 object IDs, extracted features have 25 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 10, 11, 13], only-in-features=[257, 514, 1028, 1285, 1542, 1799, 2056, 2570, 2827, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| SARCO219_T2   | C6-3       | source_mask_object_id_mismatch | Nucleocentric: mask has 26 object IDs, extracted features have 25 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 10, 11, 13], only-in-features=[257, 514, 1028, 1285, 1542, 1799, 2056, 2570, 2827, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug     |
| SARCO219_T2   | D11-1      | source_mask_object_id_mismatch | Cell: mask has 131 object IDs, extracted features have 131 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug              |
| SARCO219_T2   | D11-1      | source_mask_object_id_mismatch | Cytoplasm: mask has 130 object IDs, extracted features have 130 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO219_T2   | D11-1      | source_mask_object_id_mismatch | Nuclei: mask has 131 object IDs, extracted features have 131 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| SARCO219_T2   | D11-1      | source_mask_object_id_mismatch | Nucleocentric: mask has 131 object IDs, extracted features have 131 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug     |
| SARCO219_T2   | D2-2       | source_mask_object_id_mismatch | Cell: mask has 103 object IDs, extracted features have 98 — only-in-mask=[2, 4, 6, 7, 8, 9, 10, 12, 13, 15], only-in-features=[514, 1028, 1799, 2056, 2313, 2570, 3084, 3341, 3855, 4112] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug           |
| SARCO219_T2   | D2-2       | source_mask_object_id_mismatch | Cytoplasm: mask has 102 object IDs, extracted features have 97 — only-in-mask=[2, 4, 6, 7, 9, 10, 12, 13, 15, 16], only-in-features=[514, 1028, 1799, 2313, 2570, 3084, 3341, 3855, 4112, 4369] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug     |
| SARCO219_T2   | D2-2       | source_mask_object_id_mismatch | Nuclei: mask has 103 object IDs, extracted features have 98 — only-in-mask=[2, 4, 6, 7, 8, 9, 10, 12, 13, 15], only-in-features=[514, 1028, 1799, 2056, 2313, 2570, 3084, 3341, 3855, 4112] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO219_T2   | D2-2       | source_mask_object_id_mismatch | Nucleocentric: mask has 103 object IDs, extracted features have 98 — only-in-mask=[2, 4, 6, 7, 8, 9, 10, 12, 13, 15], only-in-features=[514, 1028, 1799, 2056, 2313, 2570, 3084, 3341, 3855, 4112] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug  |
| SARCO219_T2   | D6-4       | source_mask_object_id_mismatch | Cytoplasm: mask has 174 object IDs, extracted features have 172 — only-in-mask=[40, 46, 50, 55, 59, 60, 62, 63, 72, 80], only-in-features=[39, 43, 49, 54, 57, 61, 70, 79, 87, 100] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                 |
| SARCO219_T2   | D6-4       | source_mask_object_id_mismatch | Nuclei: mask has 175 object IDs, extracted features have 173 — only-in-mask=[40, 46, 50, 55, 59, 60, 62, 63, 72, 80], only-in-features=[39, 43, 49, 54, 57, 61, 70, 79, 87, 100] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                    |
| SARCO219_T2   | E10-3      | source_mask_object_id_mismatch | Cell: mask has 73 object IDs, extracted features have 70 — only-in-mask=[1, 2, 3, 4, 6, 8, 9, 10, 11, 12], only-in-features=[257, 514, 771, 1028, 1542, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| SARCO219_T2   | E10-3      | source_mask_object_id_mismatch | Cytoplasm: mask has 72 object IDs, extracted features have 69 — only-in-mask=[1, 2, 3, 4, 8, 9, 10, 11, 12, 13], only-in-features=[257, 514, 771, 1028, 2056, 2313, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO219_T2   | E10-3      | source_mask_object_id_mismatch | Nuclei: mask has 73 object IDs, extracted features have 70 — only-in-mask=[1, 2, 3, 4, 6, 8, 9, 10, 11, 12], only-in-features=[257, 514, 771, 1028, 1542, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug             |
| SARCO219_T2   | E10-3      | source_mask_object_id_mismatch | Nucleocentric: mask has 73 object IDs, extracted features have 70 — only-in-mask=[1, 2, 3, 4, 6, 8, 9, 10, 11, 12], only-in-features=[257, 514, 771, 1028, 1542, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug      |
| SARCO219_T2   | E2-3       | source_mask_object_id_mismatch | Cell: mask has 69 object IDs, extracted features have 69 — only-in-mask=[2, 3, 4, 6, 7, 8, 9, 10, 11, 12], only-in-features=[514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug              |
| SARCO219_T2   | E2-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 68 object IDs, extracted features have 68 — only-in-mask=[2, 3, 4, 6, 7, 8, 9, 10, 11, 12], only-in-features=[514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO219_T2   | E2-3       | source_mask_object_id_mismatch | Nuclei: mask has 69 object IDs, extracted features have 69 — only-in-mask=[2, 3, 4, 6, 7, 8, 9, 10, 11, 12], only-in-features=[514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| SARCO219_T2   | E2-3       | source_mask_object_id_mismatch | Nucleocentric: mask has 69 object IDs, extracted features have 69 — only-in-mask=[2, 3, 4, 6, 7, 8, 9, 10, 11, 12], only-in-features=[514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug     |
| SARCO219_T2   | E3-3       | source_mask_object_id_mismatch | Cell: mask has 75 object IDs, extracted features have 75 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                 |
| SARCO219_T2   | E3-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 74 object IDs, extracted features have 74 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| SARCO219_T2   | E3-3       | source_mask_object_id_mismatch | Nuclei: mask has 75 object IDs, extracted features have 75 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| SARCO219_T2   | E3-3       | source_mask_object_id_mismatch | Nucleocentric: mask has 75 object IDs, extracted features have 75 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| SARCO219_T2   | E5-2       | source_mask_object_id_mismatch | Cell: mask has 151 object IDs, extracted features have 150 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug              |
| SARCO219_T2   | E5-2       | source_mask_object_id_mismatch | Cytoplasm: mask has 150 object IDs, extracted features have 149 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO219_T2   | E5-2       | source_mask_object_id_mismatch | Nuclei: mask has 151 object IDs, extracted features have 150 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| SARCO219_T2   | E5-2       | source_mask_object_id_mismatch | Nucleocentric: mask has 151 object IDs, extracted features have 150 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug     |
| SARCO219_T2   | E5-4       | source_mask_object_id_mismatch | Cell: mask has 85 object IDs, extracted features have 84 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                 |
| SARCO219_T2   | E5-4       | source_mask_object_id_mismatch | Cytoplasm: mask has 83 object IDs, extracted features have 82 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| SARCO219_T2   | E5-4       | source_mask_object_id_mismatch | Nuclei: mask has 85 object IDs, extracted features have 84 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| SARCO219_T2   | E5-4       | source_mask_object_id_mismatch | Nucleocentric: mask has 85 object IDs, extracted features have 84 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| SARCO219_T2   | E6-2       | source_mask_object_id_mismatch | Cell: mask has 96 object IDs, extracted features have 95 — only-in-mask=[1, 2, 3, 4, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                |
| SARCO219_T2   | E6-2       | source_mask_object_id_mismatch | Cytoplasm: mask has 95 object IDs, extracted features have 94 — only-in-mask=[1, 2, 3, 4, 7, 8, 9, 10, 11, 12], only-in-features=[257, 514, 771, 1028, 1799, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| SARCO219_T2   | E6-2       | source_mask_object_id_mismatch | Nuclei: mask has 96 object IDs, extracted features have 95 — only-in-mask=[1, 2, 3, 4, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug              |
| SARCO219_T2   | E6-2       | source_mask_object_id_mismatch | Nucleocentric: mask has 96 object IDs, extracted features have 95 — only-in-mask=[1, 2, 3, 4, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug       |
| SARCO219_T2   | E7-4       | source_mask_object_id_mismatch | Cell: mask has 59 object IDs, extracted features have 57 — only-in-mask=[2, 5, 6, 7, 8, 9, 10, 11, 12, 13], only-in-features=[514, 1285, 1542, 1799, 2056, 2313, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| SARCO219_T2   | E7-4       | source_mask_object_id_mismatch | Cytoplasm: mask has 58 object IDs, extracted features have 56 — only-in-mask=[5, 6, 7, 8, 9, 10, 11, 12, 13, 14], only-in-features=[1285, 1542, 1799, 2056, 2313, 2570, 2827, 3084, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug     |
| SARCO219_T2   | E7-4       | source_mask_object_id_mismatch | Nuclei: mask has 59 object IDs, extracted features have 57 — only-in-mask=[2, 5, 6, 7, 8, 9, 10, 11, 12, 13], only-in-features=[514, 1285, 1542, 1799, 2056, 2313, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| SARCO219_T2   | E7-4       | source_mask_object_id_mismatch | Nucleocentric: mask has 59 object IDs, extracted features have 57 — only-in-mask=[2, 5, 6, 7, 8, 9, 10, 11, 12, 13], only-in-features=[514, 1285, 1542, 1799, 2056, 2313, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug   |
| SARCO219_T2   | F10-4      | source_mask_object_id_mismatch | Cell: mask has 138 object IDs, extracted features have 137 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug              |
| SARCO219_T2   | F10-4      | source_mask_object_id_mismatch | Cytoplasm: mask has 138 object IDs, extracted features have 136 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 2313, 2570, 2827, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO219_T2   | F10-4      | source_mask_object_id_mismatch | Nuclei: mask has 138 object IDs, extracted features have 137 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug            |
| SARCO219_T2   | F10-4      | source_mask_object_id_mismatch | Nucleocentric: mask has 138 object IDs, extracted features have 137 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug     |
| SARCO219_T2   | F3-4       | source_mask_object_id_mismatch | Cell: mask has 34 object IDs, extracted features have 34 — only-in-mask=[1, 2, 3, 6, 9, 12, 13, 14, 18, 20], only-in-features=[257, 514, 771, 1542, 2313, 3084, 3341, 3598, 4626, 5140] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug             |
| SARCO219_T2   | F3-4       | source_mask_object_id_mismatch | Cytoplasm: mask has 32 object IDs, extracted features have 32 — only-in-mask=[1, 2, 3, 6, 9, 13, 18, 20, 22, 24], only-in-features=[257, 514, 771, 1542, 2313, 3341, 4626, 5140, 5654, 6168] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| SARCO219_T2   | F3-4       | source_mask_object_id_mismatch | Nuclei: mask has 34 object IDs, extracted features have 34 — only-in-mask=[1, 2, 3, 6, 9, 12, 13, 14, 18, 20], only-in-features=[257, 514, 771, 1542, 2313, 3084, 3341, 3598, 4626, 5140] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug           |
| SARCO219_T2   | F3-4       | source_mask_object_id_mismatch | Nucleocentric: mask has 34 object IDs, extracted features have 34 — only-in-mask=[1, 2, 3, 6, 9, 12, 13, 14, 18, 20], only-in-features=[257, 514, 771, 1542, 2313, 3084, 3341, 3598, 4626, 5140] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug    |
| SARCO219_T2   | F5-1       | source_mask_object_id_mismatch | Cell: mask has 55 object IDs, extracted features have 54 — only-in-mask=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| SARCO219_T2   | F5-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 54 object IDs, extracted features have 53 — only-in-mask=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| SARCO219_T2   | F5-1       | source_mask_object_id_mismatch | Nuclei: mask has 55 object IDs, extracted features have 54 — only-in-mask=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug             |
| SARCO219_T2   | F5-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 55 object IDs, extracted features have 54 — only-in-mask=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug      |
| SARCO219_T2   | F5-4       | source_mask_object_id_mismatch | Cell: mask has 161 object IDs, extracted features have 160 — only-in-mask=[1, 4, 6, 7, 8, 9, 10, 11, 12, 13], only-in-features=[257, 514, 1285, 1799, 2056, 2313, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug           |
| SARCO219_T2   | F5-4       | source_mask_object_id_mismatch | Cytoplasm: mask has 160 object IDs, extracted features have 159 — only-in-mask=[1, 4, 6, 7, 9, 10, 11, 12, 13, 14], only-in-features=[257, 514, 1285, 1799, 2056, 2570, 2827, 3084, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug     |
| SARCO219_T2   | F5-4       | source_mask_object_id_mismatch | Nuclei: mask has 161 object IDs, extracted features have 160 — only-in-mask=[1, 4, 6, 7, 8, 9, 10, 11, 12, 13], only-in-features=[257, 514, 1285, 1799, 2056, 2313, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO219_T2   | F5-4       | source_mask_object_id_mismatch | Nucleocentric: mask has 161 object IDs, extracted features have 160 — only-in-mask=[1, 4, 6, 7, 8, 9, 10, 11, 12, 13], only-in-features=[257, 514, 1285, 1799, 2056, 2313, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug  |
| SARCO219_T2   | F9-1       | source_mask_object_id_mismatch | Cell: mask has 101 object IDs, extracted features have 98 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 11, 12], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| SARCO219_T2   | F9-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 100 object IDs, extracted features have 97 — only-in-mask=[1, 2, 3, 4, 7, 8, 9, 11, 12, 13], only-in-features=[257, 514, 771, 1028, 1799, 2056, 2313, 2570, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO219_T2   | F9-1       | source_mask_object_id_mismatch | Nuclei: mask has 101 object IDs, extracted features have 98 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 11, 12], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug             |
| SARCO219_T2   | F9-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 101 object IDs, extracted features have 98 — only-in-mask=[1, 2, 3, 4, 5, 7, 8, 9, 11, 12], only-in-features=[257, 514, 771, 1028, 1285, 1799, 2056, 2313, 2570, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug      |
| SARCO219_T2   | F9-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 110 object IDs, extracted features have 108 — only-in-mask=[101, 109, 118, 121, 125, 131], only-in-features=[102, 111, 119, 126] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                |
| SARCO219_T2   | F9-3       | source_mask_object_id_mismatch | Nuclei: mask has 111 object IDs, extracted features have 109 — only-in-mask=[101, 109, 118, 121, 125, 131], only-in-features=[102, 111, 119, 126] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                   |
| SARCO219_T2   | G11-2      | source_mask_object_id_mismatch | Cell: mask has 130 object IDs, extracted features have 131 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 1285, 1542, 1799, 2056, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug             |
| SARCO219_T2   | G11-2      | source_mask_object_id_mismatch | Cytoplasm: mask has 129 object IDs, extracted features have 130 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 1285, 1542, 1799, 2056, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| SARCO219_T2   | G11-2      | source_mask_object_id_mismatch | Nuclei: mask has 130 object IDs, extracted features have 131 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 1285, 1542, 1799, 2056, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug           |
| SARCO219_T2   | G11-2      | source_mask_object_id_mismatch | Nucleocentric: mask has 130 object IDs, extracted features have 131 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 1285, 1542, 1799, 2056, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug    |
| SARCO219_T2   | G11-4      | source_mask_object_id_mismatch | Cell: mask has 100 object IDs, extracted features have 97 — only-in-mask=[3, 4, 6, 8, 9, 10, 11, 12, 13, 14], only-in-features=[514, 1028, 1542, 2056, 2313, 2570, 2827, 3084, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| SARCO219_T2   | G11-4      | source_mask_object_id_mismatch | Cytoplasm: mask has 98 object IDs, extracted features have 95 — only-in-mask=[3, 4, 6, 8, 9, 10, 11, 12, 13, 14], only-in-features=[514, 1028, 1542, 2056, 2313, 2570, 2827, 3084, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug      |
| SARCO219_T2   | G11-4      | source_mask_object_id_mismatch | Nuclei: mask has 100 object IDs, extracted features have 97 — only-in-mask=[3, 4, 6, 8, 9, 10, 11, 12, 13, 14], only-in-features=[514, 1028, 1542, 2056, 2313, 2570, 2827, 3084, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| SARCO219_T2   | G11-4      | source_mask_object_id_mismatch | Nucleocentric: mask has 100 object IDs, extracted features have 97 — only-in-mask=[3, 4, 6, 8, 9, 10, 11, 12, 13, 14], only-in-features=[514, 1028, 1542, 2056, 2313, 2570, 2827, 3084, 3341, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug |
| SARCO219_T2   | G4-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 66 object IDs, extracted features have 65 — only-in-mask=[27], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                              |
| SARCO219_T2   | G4-1       | source_mask_object_id_mismatch | Nuclei: mask has 67 object IDs, extracted features have 66 — only-in-mask=[27], only-in-features=[] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                                 |
| SARCO219_T2   | G5-2       | source_mask_object_id_mismatch | Cell: mask has 55 object IDs, extracted features have 52 — only-in-mask=[1, 2, 4, 5, 7, 8, 9, 10, 11, 13], only-in-features=[257, 514, 771, 1285, 1542, 1799, 2313, 2570, 2827, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| SARCO219_T2   | G5-2       | source_mask_object_id_mismatch | Cytoplasm: mask has 55 object IDs, extracted features have 51 — only-in-mask=[1, 2, 4, 5, 7, 8, 9, 10, 11, 13], only-in-features=[257, 514, 1285, 1542, 1799, 2313, 2570, 2827, 3598, 3855] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO219_T2   | G5-2       | source_mask_object_id_mismatch | Nuclei: mask has 55 object IDs, extracted features have 52 — only-in-mask=[1, 2, 4, 5, 7, 8, 9, 10, 11, 13], only-in-features=[257, 514, 771, 1285, 1542, 1799, 2313, 2570, 2827, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug             |
| SARCO219_T2   | G5-2       | source_mask_object_id_mismatch | Nucleocentric: mask has 55 object IDs, extracted features have 52 — only-in-mask=[1, 2, 4, 5, 7, 8, 9, 10, 11, 13], only-in-features=[257, 514, 771, 1285, 1542, 1799, 2313, 2570, 2827, 3598] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug      |
| SARCO219_T2   | G8-2       | source_mask_object_id_mismatch | Cytoplasm: mask has 63 object IDs, extracted features have 63 — only-in-mask=[44], only-in-features=[73] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                            |
| SARCO219_T2   | G8-2       | source_mask_object_id_mismatch | Nuclei: mask has 64 object IDs, extracted features have 64 — only-in-mask=[44], only-in-features=[73] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                                                                               |
| SARCO219_T2   | G9-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 143 object IDs, extracted features have 143 — only-in-mask=[54, 59, 62, 71, 83, 95, 100, 102, 106, 108], only-in-features=[52, 55, 60, 66, 72, 84, 101, 104, 107, 110] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| SARCO219_T2   | G9-1       | source_mask_object_id_mismatch | Nuclei: mask has 145 object IDs, extracted features have 145 — only-in-mask=[54, 59, 62, 71, 83, 100, 102, 106, 108], only-in-features=[55, 60, 66, 72, 84, 101, 104, 107, 110] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                     |
| SARCO361_T1   | C2-7       | source_mask_object_id_mismatch | Cell: mask has 15 object IDs, extracted features have 15 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| SARCO361_T1   | C2-7       | source_mask_object_id_mismatch | Cytoplasm: mask has 14 object IDs, extracted features have 14 — only-in-mask=[1, 2, 5, 6, 7, 8, 9, 10, 11, 12], only-in-features=[257, 514, 1285, 1542, 1799, 2056, 2313, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug         |
| SARCO361_T1   | C2-7       | source_mask_object_id_mismatch | Nuclei: mask has 15 object IDs, extracted features have 15 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug             |
| SARCO361_T1   | C2-7       | source_mask_object_id_mismatch | Nucleocentric: mask has 15 object IDs, extracted features have 15 — only-in-mask=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[257, 514, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug      |
| SARCO361_T1   | C9-3       | source_mask_object_id_mismatch | Cell: mask has 8 object IDs, extracted features have 8 — only-in-mask=[1, 2, 3, 4, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                    |
| SARCO361_T1   | C9-3       | source_mask_object_id_mismatch | Cytoplasm: mask has 7 object IDs, extracted features have 7 — only-in-mask=[1, 2, 3, 4, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                        |
| SARCO361_T1   | C9-3       | source_mask_object_id_mismatch | Nuclei: mask has 8 object IDs, extracted features have 8 — only-in-mask=[1, 2, 3, 4, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                  |
| SARCO361_T1   | C9-3       | source_mask_object_id_mismatch | Nucleocentric: mask has 8 object IDs, extracted features have 8 — only-in-mask=[1, 2, 3, 4, 8, 9, 10, 11], only-in-features=[257, 514, 771, 1028, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                           |
| SARCO361_T1   | D10-4      | source_mask_object_id_mismatch | Cell: mask has 15 object IDs, extracted features have 15 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 10, 11, 12], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| SARCO361_T1   | D10-4      | source_mask_object_id_mismatch | Cytoplasm: mask has 14 object IDs, extracted features have 14 — only-in-mask=[1, 2, 4, 5, 6, 7, 10, 11, 12, 13], only-in-features=[257, 514, 1028, 1285, 1542, 1799, 2570, 2827, 3084, 3341] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| SARCO361_T1   | D10-4      | source_mask_object_id_mismatch | Nuclei: mask has 15 object IDs, extracted features have 15 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 10, 11, 12], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug             |
| SARCO361_T1   | D10-4      | source_mask_object_id_mismatch | Nucleocentric: mask has 15 object IDs, extracted features have 15 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 10, 11, 12], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2570, 2827, 3084] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug      |
| SARCO361_T1   | D2-1       | source_mask_object_id_mismatch | Cell: mask has 20 object IDs, extracted features have 19 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                |
| SARCO361_T1   | D2-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 19 object IDs, extracted features have 18 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug           |
| SARCO361_T1   | D2-1       | source_mask_object_id_mismatch | Nuclei: mask has 20 object IDs, extracted features have 19 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug              |
| SARCO361_T1   | D2-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 20 object IDs, extracted features have 19 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 10, 11], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug       |
| SARCO361_T1   | D6-1       | source_mask_object_id_mismatch | Cell: mask has 25 object IDs, extracted features have 25 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                 |
| SARCO361_T1   | D6-1       | source_mask_object_id_mismatch | Cytoplasm: mask has 24 object IDs, extracted features have 24 — only-in-mask=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11], only-in-features=[514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570, 2827] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug          |
| SARCO361_T1   | D6-1       | source_mask_object_id_mismatch | Nuclei: mask has 25 object IDs, extracted features have 25 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug               |
| SARCO361_T1   | D6-1       | source_mask_object_id_mismatch | Nucleocentric: mask has 25 object IDs, extracted features have 25 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056, 2313, 2570] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug        |
| SARCO361_T1   | E10-2      | source_mask_object_id_mismatch | Cell: mask has 8 object IDs, extracted features have 8 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                      |
| SARCO361_T1   | E10-2      | source_mask_object_id_mismatch | Cytoplasm: mask has 7 object IDs, extracted features have 7 — only-in-mask=[2, 3, 4, 5, 6, 7, 8], only-in-features=[514, 771, 1028, 1285, 1542, 1799, 2056] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                         |
| SARCO361_T1   | E10-2      | source_mask_object_id_mismatch | Nuclei: mask has 8 object IDs, extracted features have 8 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                                    |
| SARCO361_T1   | E10-2      | source_mask_object_id_mismatch | Nucleocentric: mask has 8 object IDs, extracted features have 8 — only-in-mask=[1, 2, 3, 4, 5, 6, 7, 8], only-in-features=[257, 514, 771, 1028, 1285, 1542, 1799, 2056] — the segmentation mask on disk does not match the extracted features, so the source mask is the likely cause (re-segmented/overwritten after features were last extracted) rather than a merge/extraction bug                             |

## Well-FOVs with the most issues (top 30)

| patient     | well_fov   |   n_issues | file_count_mismatch   | object_ids_aligned_across_compartments   | misalignment_root_cause   | organoid_profiles_empty   |   n_source_mask_mismatches |   n_object_id_exceeds_threshold |
|:------------|:-----------|-----------:|:----------------------|:-----------------------------------------|:--------------------------|:--------------------------|---------------------------:|--------------------------------:|
| NF0018_T6   | G9-1       |         11 | False                 | False                                    | image_level               | True                      |                          4 |                               4 |
| SARCO219_T2 | E3-3       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | F5-4       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| NF0018_T6   | F8-1       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | G11-4      |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| NF0030_T1   | D3-1       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | D2-2       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | E7-4       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| NF0021_T1   | E2-1       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | G11-2      |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO361_T1 | D2-1       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO361_T1 | E10-2      |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | F10-4      |         10 | False                 | False                                    | feature_extraction_level  | False                     |                          4 |                               4 |
| SARCO361_T1 | D6-1       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| NF0030_T1   | C7-2       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO361_T1 | C2-7       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| NF0018_T6   | F8-4       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | E10-3      |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | F9-1       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | E2-3       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| NF0021_T1   | G11-7      |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| NF0035_T1   | G7-3       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | C3-1       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO361_T1 | C9-3       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | D11-1      |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | E5-2       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | C6-3       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| NF0016_T1   | E4-1       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO361_T1 | D10-4      |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |
| SARCO219_T2 | F3-4       |         10 | False                 | False                                    | image_level               | False                     |                          4 |                               4 |

## Notes

- Full per-issue detail is in `well_fov_feature_merge_issues_all_patients.csv`.
- An empty Organoid profile (no files, or a merge with zero rows) is tracked
  separately and does NOT count as an issue — some well-FOVs legitimately have
  no organoids. An Organoid merge that fails outright because one input file is
  an empty (0-row) profile IS still counted, since that reflects an actual
  merge bug rather than a legitimately-empty well-FOV.
- The source mask consistency check reads segmentation mask TIFFs directly (not 
  through the extracted features) to determine whether an object-ID discrepancy 
  originates from the mask itself vs. from feature extraction/merging.
- The root-cause check goes one step further for `object_id_misalignment` cases
  specifically: it compares the implicated compartments' masks directly against
  EACH OTHER (not just each mask against its own features), which is the only way
  to tell an image-level misalignment (masks disagree with each other) apart from
  a feature-extraction-level one (masks agree, but the feature tables don't).
- `Organoid` is excluded from the cross-compartment object-ID alignment check; it's
  a distinct feature space (one row per whole organoid) with its own object-ID
  space that isn't expected to match the single-cell compartments.
- Each patient's expected file count is that patient's own mode file count across
  its well-FOVs (excluding zero-file well-FOVs) — patients can have different
  channel/feature-type combinations, so a single global expected count doesn't
  generalize across patients.